# Recovering AHA 2017 coronary categories from Chinese echocardiography reports
## Coronary reclassification notebook v1 (Wuhan Children's Hospital KD registry, 30-day re-export)

**Problem.** Sonographers write 扩张 ("dilation") or 瘤 ("aneurysm"), and clinicians code 冠状动脉扩张 / 冠状动脉动脉瘤, but
neither follows the AHA 2017 Z-score categories. The frozen master v1.3 inherits this when no Z-score is available
(`severity_source = qualitative_*`). This notebook recovers, for every echo report and every child, the AHA category that the
numbers support, states how each value was obtained, and measures how well each step works.

**Design (locked; change here and re-run).**
* AHA 2017 categories from the maximal coronary Z-score of the report: 0 no involvement (Z < 2); 1 dilation only (2 to < 2.5);
  2 small aneurysm (2.5 to < 5); 3 medium (5 to < 10); 4 large/giant (Z ≥ 10 or internal diameter ≥ 8 mm).
* **Tier 1** – Z written next to the diameter in the report. **Tier 2** – Z computed from a written diameter with the echo laboratory's
  own equations, reconstructed here from the reports that print diameter, Z, height and weight together (Stevenson BSA;
  mean = a + b·√BSA + c·BSA; constant SD per vessel). BSA source: same report → nearest growth record ≤ 90 d → nearest record
  projected with the cohort's age–sex BSA medians → the age–sex median itself. **Tier 3** – diameters/Z transcribed into a discharge
  summary or outpatient note and linked to the report by date. **Tier 4** – calibrated wording model (multinomial logistic regression on
  text features only, trained on reports with complete numbers) for reports without numbers.
* If the report calls a segment abnormal but gives no diameter for it and nothing measured on that side reaches Z ≥ 2.5, the numeric
  category is only a floor; the wording model then gives the distribution above the floor.
* Child level: maximum over the reports in a window (in hospital; ≤ 30 d after discharge; whole follow-up). Report distributions are
  combined as a comonotone maximum (CDF of the maximum = minimum of the CDFs), which is conservative for repeated reports of one lesion.
* Validation: blinded manual extraction audits (development and independent samples, `audit/`), linkage accuracy of note harvesting,
  patient-grouped cross-validation and temporal validation of the wording model, external anchors (warfarin/clopidogrel, discharge
  coding), and a crosswalk to the frozen master v1.3 (reference only).

**How to run.** Put this notebook in the folder with `川崎病重新导出30天_1/2/3.xlsx` (optionally `KD_Infant_Master_v1.3.xlsx` and the
`audit/` folder) and run all cells. Outputs go to `./output_kd_coronary_aha_v1/`. Every number is computed; nothing is typed in.

In [ ]:
import os, re, glob, json, time, warnings, shutil, hashlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import least_squares, minimize
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 80)

DATA_DIR = '.'
OUT_DIR = os.path.join(DATA_DIR, 'output_kd_coronary_aha_v1')
CACHE = os.path.join(OUT_DIR, 'cache')
os.makedirs(CACHE, exist_ok=True)
for alt in ['output_kd_echo_surveillance_v3', 'output_kd_echo_surveillance_v2', 'output_kd_echo_surveillance', 'output_kd30_screening']:
    alt = os.path.join(DATA_DIR, alt, 'cache')
    if os.path.isdir(alt):
        for f in glob.glob(os.path.join(alt, '*.parquet')):
            if not os.path.exists(os.path.join(CACHE, os.path.basename(f))):
                shutil.copy(f, CACHE)

plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 11,
                     'axes.labelsize': 12, 'axes.labelweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.grid': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
BLACK, DGREY, MGREY, LGREY = '#000000', '#4d4d4d', '#8c8c8c', '#c8c8c8'
CAT_COL = ['#ffffff', '#d9d9d9', '#a6a6a6', '#636363', '#000000']          # AHA 0..4, white -> black
CAT_LAB = ['No involvement (Z<2)', 'Dilation only (2–<2.5)', 'Small aneurysm (2.5–<5)', 'Medium aneurysm (5–<10)', 'Large/giant (≥10 or ≥8 mm)']
CAT_SHORT = ['None', 'Dilation only', 'Small', 'Medium', 'Large/giant']

files = sorted(glob.glob(os.path.join(DATA_DIR, '川崎病重新导出30天_*.xlsx')))
assert len(files) == 3, f'expected the three workbooks in {os.path.abspath(DATA_DIR)}, found {files}'
print('workbooks:', [os.path.basename(f) for f in files])
KEY = {}          # key numbers for the results lock


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n; c = p + z**2 / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (max((c - h) / d * 100, 0.0), min((c + h) / d * 100, 100.0))


def fmt_rate(k, n):
    lo, hi = wilson(k, n)
    return f'{k}/{n} ({k/n*100:.1f}%; 95% CI {lo:.1f}–{hi:.1f})' if n else 'NA'


def panel(ax, letter):
    ax.text(-0.13, 1.04, letter, transform=ax.transAxes, fontsize=15, fontweight='bold', va='bottom', ha='left')


def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(OUT_DIR, f'{name}.{ext}'), bbox_inches='tight', facecolor='white')
    plt.close(fig)

## 1. Load sheets (cached as parquet after the first run)

In [ ]:
NEEDED = {'患者基本信息', '病案首页基本信息', '病案出院诊断编目前', '病案出院诊断归一', '儿童心脏彩超', '住院用药医嘱', '住院病历出院记录',
          '儿科入院记录', '门急诊病历'}


def safe(name):
    return re.sub(r'[^\w一-鿿]+', '_', name)


def load_sheet(name):
    p = os.path.join(CACHE, safe(name) + '.parquet')
    if os.path.exists(p):
        return pd.read_parquet(p).astype('str')
    import openpyxl
    for f in files:
        wb = openpyxl.load_workbook(f, read_only=True)
        if name in wb.sheetnames:
            ws = wb[name]; rows = ws.iter_rows(values_only=True); hdr = next(rows)
            cols = [str(c) if c is not None else f'col{i}' for i, c in enumerate(hdr)]
            df = pd.DataFrame(list(rows), columns=cols).dropna(how='all')
            for c in df.columns:
                df[c] = df[c].astype('string')
            wb.close(); df.to_parquet(p, index=False)
            return df.astype('str')
        wb.close()
    raise KeyError(name)


t0 = time.time(); T = {n: load_sheet(n) for n in sorted(NEEDED)}
print({n: T[n].shape for n in sorted(NEEDED)}); print(f'loaded in {time.time()-t0:.0f} s')


def txt(s):
    """text column with missing values as '' (cached sheets may hold NaN or the strings 'None'/'nan'/'<NA>')"""
    s = s.astype(object).where(s.notna(), '').astype(str)
    return s.where(~s.isin(['None', 'nan', '<NA>', 'NaT']), '')

## 2. Index admissions and echo phases (same definitions as the surveillance notebooks)

In [ ]:
KD_PAT = r'川崎|皮肤粘膜淋巴结|皮肤黏膜淋巴结|黏膜皮肤淋巴结|粘膜皮肤淋巴结|kawasaki'
adm = T['病案首页基本信息'].copy()
adm['入院日期'] = pd.to_datetime(adm['入院日期'], errors='coerce'); adm['出院日期'] = pd.to_datetime(adm['出院日期'], errors='coerce')
DATA_END = adm['出院日期'].max().normalize()
dx = T['病案出院诊断编目前'].copy()
dx['is_kd'] = dx['诊断疾病名称'].str.contains(KD_PAT, case=False, na=False)
adm['kd_dx'] = adm['科研就诊编号'].map(dx.groupby('科研就诊编号')['is_kd'].any()).fillna(False).astype(bool)
adm = adm.sort_values(['科研患者编号', '入院日期'])
idx = adm[adm['kd_dx']].groupby('科研患者编号').first().reset_index()[['科研患者编号', '科研就诊编号', '入院日期', '出院日期']]
idx.columns = ['科研患者编号', 'index_enc', 'idx_adm', 'idx_dis']
idx['year'] = idx['idx_adm'].dt.year
pt = T['患者基本信息'].drop_duplicates('科研患者编号').copy()
pt['dob'] = pd.to_datetime(pt['出生日期'], errors='coerce')
PT = pt[['科研患者编号', 'dob', '性别']].copy()
print('index KD admissions:', len(idx), '| years', int(idx['year'].min()), '–', int(idx['year'].max()), '| data end', DATA_END.date())

e = T['儿童心脏彩超'].copy()
e['儿童超声心动图所见'] = txt(e['儿童超声心动图所见']); e['儿童超声心动图结论'] = txt(e['儿童超声心动图结论'])
e['检查时间'] = pd.to_datetime(e['检查时间'], errors='coerce')
e = e.merge(idx[['科研患者编号', 'index_enc', 'idx_adm', 'idx_dis', 'year']].rename(columns={'year': 'idx_year'}), on='科研患者编号', how='left')
e['d_adm'] = (e['检查时间'] - e['idx_adm'].dt.normalize()).dt.days; e['d_dis'] = (e['检查时间'] - e['idx_dis'].dt.normalize()).dt.days
e['phase'] = np.select([e['index_enc'].isna(), e['d_adm'] < 0, (e['d_adm'] >= 0) & (e['d_dis'] <= 0), (e['d_dis'] > 0) & (e['d_dis'] <= 30),
                        (e['d_dis'] > 30) & (e['d_dis'] <= 90), (e['d_dis'] > 90) & (e['d_dis'] <= 365), e['d_dis'] > 365],
                       ['no_kd_index', 'pre_index', 'in_index', 'post30', 'd31_90', 'd91_365', 'gt365'], 'other')
e = e.sort_values(['科研患者编号', '检查时间']).reset_index(drop=True)
e['yr'] = e['检查时间'].dt.year


def report_key(pid, t, f):
    return f"{pid}|{pd.Timestamp(t).strftime('%Y-%m-%d %H:%M:%S')}|{hashlib.md5(str(f).encode('utf-8')).hexdigest()[:10]}"


e['report_key'] = [report_key(a, b, c) for a, b, c in zip(e['科研患者编号'], e['检查时间'], e['儿童超声心动图所见'])]
print('echo reports:', len(e), '| phases:', e['phase'].value_counts().to_dict())

## 3. Sonographer wording (rule-based reading used in the surveillance notebooks; audited 200/200)

In [ ]:
LEGEND_W = re.compile(r'[（(]?\s*z\s*值\s*绝对值[^）)]*[）)]?', re.I)
LEGEND_W2 = re.compile(r'(<|＜)?\s*2\.5\s*或\s*(>|＞)?\s*-?2\.5[^；;。\n]*|2\.5\s*[-—–~]\s*5\s*[：:]\s*小型冠脉瘤|5\s*[-—–~]\s*10\s*[：:]\s*中型冠脉瘤|(>|＞)\s*10\s*[：:]\s*大型冠脉瘤|正常[；;]?\s*2\.5', re.I)
NEG_W = re.compile(r'未见|不宽|无明显|无扩张|未扩张|无增宽|未增宽|正常|未探及|尚可|未及|恢复|消失')
ANEU_W = re.compile(r'瘤'); DIL_W = re.compile(r'扩张|增宽|增粗|扩大|稍宽|略宽|偏宽')
CORON_W = re.compile(r'冠|前降支|回旋支|旋支|左主干|LMCA|LCA|LAD|LCX|RCA', re.I)
OTHER_W = re.compile(r'肺动脉|主动脉|静脉|瓣|房间隔|室间隔|导管|心包')
VEIN_W = re.compile(r'冠状静脉|冠状窦|冠脉静脉'); UPPER_W = re.compile(r'正常值?上限|正常高限|上限值|临界')


def wclean(t):
    return LEGEND_W2.sub('', LEGEND_W.sub('', t)).replace('肺动脉主干', '肺动脉干') if isinstance(t, str) else ''


def coronary_lines(t):
    out = []
    for l in [s.strip() for s in re.split(r'[\n；;。]', t) if s.strip()]:
        if CORON_W.search(l):
            l2 = '，'.join(c for c in re.split(r'[，,]', l) if not VEIN_W.search(c))
            if CORON_W.search(l2):
                out.append(l2)
    return out


def status_from_text(t):
    lines = coronary_lines(t)
    if not lines:
        return np.nan, False
    st, upper = 0, False
    for l in lines:
        for c in re.split(r'[，,、]', l):
            if not c.strip():
                continue
            if OTHER_W.search(c) and not CORON_W.search(c):
                continue
            if ANEU_W.search(c) and not re.search(r'未见.*瘤|无.*瘤|瘤.*未见|瘤.*消失', c):
                st = max(st, 2)
            elif DIL_W.search(c) and not NEG_W.search(c):
                st = max(st, 1)
            elif UPPER_W.search(c):
                upper = True
        if ANEU_W.search(l) and not re.search(r'未见[^，,]*瘤|瘤[^，,]*消失', l):
            st = max(st, 2)
    return st, upper


rows = []
for f, c in zip(e['儿童超声心动图所见'].tolist(), e['儿童超声心动图结论'].tolist()):
    st_c, up_c = status_from_text(wclean(c)); st_f, up_f = status_from_text(wclean(f))
    rows.append(dict(cor_status=st_c if not np.isnan(st_c) else st_f, upper_normal=bool(up_c or up_f)))
e = pd.concat([e, pd.DataFrame(rows, index=e.index)], axis=1)
WORD = {0: 'normal', 1: 'dilation', 2: 'aneurysm'}
print('wording:', e['cor_status'].map(WORD).value_counts(dropna=False).to_dict())

## 4. Per-vessel measurement extractor
For every report and vessel (LMCA, LAD, LCX, RCA): every luminal diameter with its paired Z (if written) and segment label; unpaired
Z values; whether the vessel is described as abnormal; whether an abnormal segment has no diameter of its own. Lengths, ranges,
distances, wall thickness, thrombus/echo sizes, jet widths, pressure gradients, valve cusps (右冠瓣) and the printed Z legend are excluded;
aneurysm dimensions 'A×B（长径×内径）' contribute the internal diameter; in-order lists ('…分别为…') are resolved only when the counts match.
Height/weight printed in the report are captured for BSA.

In [ ]:
_RCA = r'(?:右侧?冠状动脉|右侧?冠脉|右侧?冠动脉)(?![瓣窦])|右侧?冠(?![瓣窦状脉动])|后降支|RCA|RAC'
ANCHOR = re.compile(
    r'(?P<LAD>左冠状动脉前降支|左冠脉前降支|左前降支|前降支|前降(?!低)|LAD)'
    r'|(?P<LCX>左冠状动脉回旋支|左冠脉回旋支|左回旋支|回旋支|旋支|LCX|LCx)'
    r'|(?P<BOTH>左、右冠状动脉|左右冠状动脉|左、右冠脉|左右冠脉|双侧冠状动脉|双侧冠脉)'
    r'|(?P<LMCA>左冠?侧?冠状动脉主干|左侧?冠脉主干|左侧?冠动脉主干|左主干|LMCA|LCA|LAC|左冠?侧?冠状动脉(?![瓣窦])|左侧?冠脉(?![瓣窦])|左侧?冠动脉(?![瓣窦])|左侧?冠(?![瓣窦状脉动]))'
    rf'|(?P<RCA>{_RCA})')
VESSELS = ['LMCA', 'LAD', 'LCX', 'RCA']
ZPAT = re.compile(r'[Zz]\s*(?:值|score|分数)?\s*[：:＝=]?\s*[+＋]?\s*(-?\d+(?:\.\d+)?)')
MM = re.compile(r'(\d+(?:\.\d+)?)\s*(?:mm|MM|Mm|毫米)(?![Hh][Gg])')
# numbers that are not luminal diameters
LENGTH_WORDS = re.compile(r'范围|长约|长度|长径|可见长|显示段长|段长|距开口|距离|距|束宽|分流|回声团|光团|光斑|强回声|血栓|厚约|厚度|厚|瓣口|瓣环|压差')
SEG = re.compile(r'近中段|近、中段|近端|近段|起始|开口|中远段|中段|远段|远端|分叉|主干|全程')
STOP = re.compile(r'[。\n]|(?<![\d.])\d+、')
ABN = re.compile(r'扩张|增宽|增粗|瘤样|瘤状|串珠|膨大|冠脉瘤|动脉瘤|瘤形成|呈瘤')
NEG = re.compile(r'未见|未探及|无明显|不宽|不扩张|未扩张|无扩张|不增宽|未增宽|正常|消失|恢复|不明显|未显示|排除')
DESCR = re.compile(r'扩张段|以远|瘤体|瘤壁|瘤内|瘤腔')   # clauses describing an already-named dilated segment (length, wall, distal part)
HEIGHT = [re.compile(r'身[高长]\s*[:：]?\s*(\d{2,3}(?:\.\d+)?)\s*(?:cm|CM|厘米)?'), re.compile(r'Height\s*[:：]?\s*(\d{2,3}(?:\.\d+)?)\s*cm', re.I)]
WEIGHT = [re.compile(r'体重\s*[:：]?\s*(\d{1,3}(?:\.\d+)?)\s*(?:kg|KG|Kg|公斤)?'), re.compile(r'Weight\s*[:：]?\s*(\d{1,3}(?:\.\d+)?)\s*kg', re.I)]
LEGEND = re.compile(r'[（(]?\s*z\s*值\s*绝对值[^）)]*[）)]?|2\.5\s*[-—–~]\s*5\s*[：:]\s*小型冠脉瘤|5\s*[-—–~]\s*10\s*[：:]\s*中型冠脉瘤|[>＞]\s*10\s*[：:]\s*大型冠脉瘤', re.I)
SIZE_WORDS = [(4, re.compile(r'巨大|巨型|大型')), (3, re.compile(r'中型|中等大小')), (2, re.compile(r'小型'))]
MORPH = {
    'saccular': re.compile(r'囊状|囊样|球状|球形'),
    'fusiform': re.compile(r'梭形|梭状|管状扩张'),
    'beaded': re.compile(r'串珠'),
    'aneurysm_word': re.compile(r'瘤'),
    'thrombus': re.compile(r'血栓|附壁.{0,6}回声|团状.{0,4}回声|回声团'),
    'giant_word': re.compile(r'巨大|巨型'),
}
STRUCT = (r'(?:主动脉瓣环|瓣环|升主动脉|主动脉窦部|主动脉根部|主动脉|肺动脉干|左肺动脉|右肺动脉|肺动脉|'
          r'左冠状动脉主干|左冠脉主干|左主干|LMCA|LCA|左冠状动脉前降支|左冠脉前降支|左前降支|前降支|LAD|'
          r'左冠状动脉回旋支|左冠脉回旋支|左回旋支|回旋支|旋支|LCX|LCx|右冠状动脉|右冠脉|右冠|RCA|左冠状动脉|左冠脉)')
VAL = r'\d+(?:\.\d+)?\s*(?:mm|MM|毫米)?'
LIST_PAT = re.compile(rf'({STRUCT}(?:\s*[、，,及和与]\s*{STRUCT})+)\s*(?:起始段|近段)?\s*(?:内径|直径|宽度)?\s*(?:分别)?\s*(?:为|约|是|约为|为约)?\s*[：:]?\s*'
                      rf'({VAL}(?:\s*[、，,及和与]\s*{VAL})+)')
DIMS = re.compile(r'(\d+(?:\.\d+)?)\s*(?:mm|MM|m)?\s*[×xX*]\s*(\d+(?:\.\d+)?)\s*(?:mm|MM)\s*(?:[（(]\s*(长径|内径|宽径|长|宽)\s*[×xX*]\s*(长径|内径|宽径|长|宽)\s*[）)])?')
BOTH_PAT = re.compile(r'(?:内径|直径)?\s*分别\s*(?:约|为|约为)?\s*[：:]?\s*(\d+(?:\.\d+)?)\s*(?:mm|MM)?\s*[及和、,，与]\s*(\d+(?:\.\d+)?)\s*(?:mm|MM)'
                      r'|(?:内径|直径)\s*(?:约|为|约为)?\s*[：:]?\s*(\d+(?:\.\d+)?)\s*(?:mm|MM)?\s*[及和、与]\s*(\d+(?:\.\d+)?)\s*(?:mm|MM)')


class _A:
    """minimal stand-in for a regex match (synthetic anchor)"""
    def __init__(self, g, a, b):
        self.lastgroup, self._a, self._b = g, a, b
    def start(self):
        return self._a
    def end(self):
        return self._b


def _clean(t):
    if not isinstance(t, str):
        return ''
    t = LEGEND.sub(' ', t)
    t = t.replace('肺动脉主干', '肺动脉干').replace('　', ' ')
    # re-join clauses broken by a line feed in the middle of a sentence (e.g. '较大内径约\n   4.4mm')
    t = re.sub(r'(?<=[^。；;\n])\n\s*(?=\d+(?:\.\d+)?\s*(?:mm|MM))', '', t)
    t = re.sub(r'(?<=[，,、])[ \t]*\n\s*(?!\d+\s*[、.．])', '', t)
    t = re.sub(r'(?<=[：:=＝])[ \t]*\n\s*(?=[+＋-]?\d)', '', t)
    t = re.sub(r'(?:(?<=冠脉)|(?<=动脉)|(?<=前降支)|(?<=回旋支)|(?<=旋支)|(?<=LAD)|(?<=LCX)|(?<=RCA)|(?<=LCA))[ \t]*\n\s*(?!\d+\s*[、.．])', '', t)
    return t


def _diameters(scope):
    """(position, value, segment_label, is_aneurysm_dim) for every luminal diameter in the scope."""
    out = []
    blank = scope
    # aneurysm dimensions 'A mm x B mm (long x inner)' -> inner diameter; unlabelled aneurysm dims -> smaller value
    for m in DIMS.finditer(scope):
        pre = scope[max(0, m.start() - 14):m.start()]
        cut = max(pre.rfind('，'), pre.rfind(','), pre.rfind('；'), pre.rfind(';'))
        pre_c = pre[cut + 1:] if cut >= 0 else pre
        a, b = float(m.group(1)), float(m.group(2))
        v = None
        if m.group(3):
            lab = [m.group(3), m.group(4)]
            if '内径' in lab or '宽径' in lab or '宽' in lab:
                v = a if lab[0] in ('内径', '宽径', '宽') else b
        elif re.search(r'瘤', pre_c) and not re.search(r'回声|光团|光斑|血栓', pre_c):
            v = min(a, b)
        if v is not None and 0.8 <= v <= 30:
            out.append((m.start(), v, 'aneurysm', True))
        blank = blank[:m.start()] + ' ' * (m.end() - m.start()) + blank[m.end():]
    for m in MM.finditer(blank):
        pre = blank[max(0, m.start() - 10):m.start()]
        cut = max(pre.rfind('，'), pre.rfind(','), pre.rfind('；'), pre.rfind(';'))
        pre_c = pre[cut + 1:] if cut >= 0 else pre
        post = blank[m.end():m.end() + 4]
        if LENGTH_WORDS.search(pre_c) or re.match(r'\s*[（(]\s*长|\s*(?:范围|长)', post) or re.match(r'\s*[×xX*]', post) or re.search(r'[×xX*]\s*$', pre_c):
            if not re.match(r'\s*[（(]\s*(?:宽|内径)', blank[m.end():m.end() + 6]):
                continue
        v = float(m.group(1))
        if 0.8 <= v <= 30:
            seg_pre = blank[max(0, m.start() - 16):m.start()]
            cut2 = max(seg_pre.rfind('，'), seg_pre.rfind(','), seg_pre.rfind('；'), seg_pre.rfind(';'))
            seg_pre = seg_pre[cut2 + 1:] if cut2 >= 0 else seg_pre
            s = SEG.findall(seg_pre)
            if not s:
                back = blank[max(0, m.start() - 30):m.start()]
                cut3 = max(back.rfind('；'), back.rfind(';'), back.rfind('。'))
                s = SEG.findall(back[cut3 + 1:] if cut3 >= 0 else back)
            out.append((m.start(), v, s[-1] if s else '', False))
    # diameters printed without a unit but directly followed by their Z ('RCA：2.3（Z值1.05）')
    for m in re.finditer(r'(?<![\d.])(\d\.\d{1,2})(?!\s*(?:mm|MM|毫米|m/s|cm))(?=\s*[（(，,]?\s*[Zz]\s*值)', blank):
        v = float(m.group(1))
        if 0.8 <= v <= 20 and not re.search(r'[Zz]\s*值?\s*[：:＝=]?\s*[+＋]?\s*-?$', blank[max(0, m.start() - 8):m.start()]):
            out.append((m.start(), v, '', False))
    return sorted(out)


def _scope_numbers(scope):
    ds = _diameters(scope)
    zs = [(m.start(), float(m.group(1))) for m in ZPAT.finditer(scope)]
    items, used = [], set()
    for zpos, zval in zs:
        if not (-8 < zval < 100):
            continue
        prev = [d for d in ds if d[0] < zpos]
        if prev:
            dpos = prev[-1][0]
            between = scope[dpos:zpos]
            if len(between) <= 25 and not re.search(r'[；;]', between):
                used.add((dpos, zval))
    paired = {p: z for p, z in used}
    zfree = [z for zpos, z in zs if -8 < z < 100 and not any(z == zz for _, zz in used)]
    for pos, v, seg, isdim in ds:
        items.append((v, paired.get(pos, np.nan), seg))
    return items, zfree


PROX = {'', '近段', '近端', '起始', '近中段', '近、中段', '主干', '全程', '分叉', 'list', 'both', 'aneurysm'}
COMPAT = {'': PROX | {'中段', '中远段'},
          '中段': {'中段', '中远段', '近中段', '近、中段', '全程', 'aneurysm'},
          '中远段': {'中段', '中远段', '远段', '远端', '全程', 'aneurysm'},
          '远段': {'远段', '远端', '中远段', '全程', 'aneurysm'},
          '开口': {'开口', '起始', '近段', '近端', ''},
          '分叉': {'分叉', '', '主干', 'aneurysm'}}
for _k in ('近段', '近端', '起始', '近中段', '近、中段', '主干', '全程'):
    COMPAT[_k] = PROX
COMPAT['远端'] = COMPAT['远段']


def _seg_label(txt):
    s = SEG.findall(txt)
    return s[0] if s else ''


def _abn_flags(scope):
    """abn_text: a non-negated abnormal assertion exists; unmeas: segment labels of abnormal assertions that carry no
    diameter of their own (neither in the clause nor in the following clauses up to the next segment change / negation /
    new assertion). Whether another measurement of the same vessel covers that segment is decided later (COMPAT)."""
    clauses = [c for c in re.split(r'[，,；;]', scope)]
    abn_text = False; unmeas = []
    for i, c in enumerate(clauses):
        if ABN.search(c) and not NEG.search(c) and not DESCR.search(c):
            abn_text = True
            has_d = bool(_diameters(c))
            j = i + 1
            while not has_d and j < len(clauses) and j <= i + 3:
                nxt = clauses[j]
                if NEG.search(nxt) or re.search(r'开口|远段|远端', nxt[:8]) or ABN.search(nxt):
                    break
                if _diameters(nxt):
                    has_d = True
                j += 1
            if not has_d:
                unmeas.append(_seg_label(c))
    return abn_text, unmeas


def _covered(items, label):
    ok = COMPAT.get(label, PROX)
    return any(seg in ok for _, _, seg in items)


def _resolve_lists(f, out):
    """'A、B、C及D内径分别为a、b、c和d mm' -> assign values to listed structures in order (count must match);
    the span is blanked in every case so that the anchor scan cannot misassign its numbers."""
    for m in list(LIST_PAT.finditer(f)):
        if 'mm' not in m.group(2).lower() and '毫米' not in m.group(2):
            continue
        names = [n for n in re.findall(STRUCT, m.group(1))]
        vals = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', m.group(2))]
        if len(names) == len(vals):
            for n, v in zip(names, vals):
                a = ANCHOR.fullmatch(n) or ANCHOR.search(n)
                if a and a.lastgroup in VESSELS and 0.8 <= v <= 30:
                    out[a.lastgroup]['items'].append((v, np.nan, 'list'))
        f = f[:m.start()] + ' ' * (m.end() - m.start()) + f[m.end():]
    return f


def extract_measurements(findings, conclusion=''):
    f = _clean(findings)
    out = {v: dict(items=[], zfree=[], abn_text=False, unmeas=[]) for v in VESSELS}
    f = _resolve_lists(f, out)
    # anchors used as landmarks ('距旋支开口约5.8mm处') do not open a new vessel scope
    anchors = [a for a in ANCHOR.finditer(f) if not re.search(r'[距近至达邻]', f[max(0, a.start() - 2):a.start()])]
    extra = []
    for mz in re.finditer(r'(?<![肺冠脉动左右])主干', f):
        if re.search(r'[距近至达邻自从]', f[max(0, mz.start() - 2):mz.start()]):
            continue          # landmark ('距主干…处'), not a measurement of the main stem
        prev = [a for a in anchors if a.end() <= mz.start()]
        between = f[prev[-1].end():mz.start()] if prev else ''
        if prev and prev[-1].lastgroup in ('LAD', 'LCX', 'BOTH', 'LMCA') and not re.search(r'[。；;\n]|\d+、|肺|主动脉', between):
            extra.append(_A('LMCA', mz.start(), mz.end()))
    anchors = sorted(anchors + extra, key=lambda a: a.start())
    for i, a in enumerate(anchors):
        vessel = a.lastgroup
        end = anchors[i + 1].start() if i + 1 < len(anchors) else len(f)
        scope = f[a.end():end]
        stop = STOP.search(scope)
        if stop:
            scope = scope[:stop.start()]
        if vessel == 'BOTH':
            m = BOTH_PAT.search(scope)
            if m:
                g = [x for x in m.groups() if x is not None]
                out['LMCA']['items'].append((float(g[0]), np.nan, 'both')); out['RCA']['items'].append((float(g[1]), np.nan, 'both'))
            at, au = _abn_flags(scope)
            if at:
                out['LMCA']['abn_text'] = True; out['RCA']['abn_text'] = True
                out['LMCA']['unmeas'] += [('LEFT', l) for l in au]; out['RCA']['unmeas'] += [('RCA', l) for l in au]
            continue
        items, zfree = _scope_numbers(scope)
        out[vessel]['items'] += items; out[vessel]['zfree'] += zfree
        at, au = _abn_flags(scope)
        out[vessel]['abn_text'] |= at; out[vessel]['unmeas'] += [(vessel, l) for l in au]
    res = {}
    for v in VESSELS:
        it = out[v]['items']
        res[f'{v}_items'] = it
        res[f'{v}_D'] = max(x[0] for x in it) if it else np.nan
        zr = [x[1] for x in it if np.isfinite(x[1])] + out[v]['zfree']
        res[f'{v}_Zrep'] = max(zr) if zr else np.nan
        res[f'{v}_Zfree'] = out[v]['zfree']
        res[f'{v}_abn_text'] = out[v]['abn_text']
        left_items = out['LMCA']['items'] + out['LAD']['items'] + out['LCX']['items']
        res[f'{v}_abn_unmeas'] = any(not _covered(left_items if tgt == 'LEFT' else out[tgt]['items'], lab) for tgt, lab in out[v]['unmeas'])
    h = w = np.nan
    for p in HEIGHT:
        m = p.search(f)
        if m:
            v = float(m.group(1))
            if 40 <= v <= 200:
                h = v; break
    for p in WEIGHT:
        m = p.search(f)
        if m:
            v = float(m.group(1))
            if 2 <= v <= 150:
                w = v; break
    res['height_cm'] = h; res['weight_kg'] = w
    concl = _clean(conclusion)
    txt_all = f + '\n' + concl
    cor_txt = '\n'.join(s for s in re.split(r'[\n；;。]', txt_all) if ANCHOR.search(s) or re.search(r'冠脉瘤|冠状动脉瘤', s))
    for k, p in MORPH.items():
        hit = False
        for s in re.split(r'[，,]', cor_txt):
            if p.search(s) and not re.search(r'未见|未探及|无明显|消失', s):
                hit = True; break
        res[f'morph_{k}'] = hit
    size = 0
    for s in re.split(r'[，,\n；;。]', cor_txt):
        if re.search(r'瘤', s) and not re.search(r'未见|未探及|无明显|消失', s):
            for lvl, p in SIZE_WORDS:
                if p.search(s):
                    size = max(size, lvl); break
    res['size_word'] = size
    # vessels called abnormal in the conclusion, and whether each has a measurement of a representative segment
    conc_abn = {}
    for cl in re.split(r'[，,；;。\n\r\t ]+|(?<!左)、(?=右侧?冠|左侧?冠|前降|回旋|LAD|LCX|RCA|LCA)', concl):
        if ABN.search(cl) and not NEG.search(cl.replace('正常值上限', '').replace('正常上限', '')):
            lab = _seg_label(cl)
            subs = [a.lastgroup for a in ANCHOR.finditer(cl)]
            left_main = '主干' in cl and not re.search(r'右侧?冠(?:状动)?脉?主干', cl) and any(x in ('LMCA', 'LAD', 'LCX', 'BOTH') for x in subs)
            tg = set()
            if any(x in ('LAD', 'LCX') for x in subs) or left_main:
                tg |= {x for x in subs if x in ('LAD', 'LCX', 'RCA')}
                if left_main:
                    tg.add('LMCA')
            else:
                for x in subs:
                    tg |= ({'LEFT', 'RCA'} if x == 'BOTH' else ({'LEFT'} if x == 'LMCA' else {x}))
            for t in tg:
                conc_abn.setdefault(t, []).append(lab)
    left_items = out['LMCA']['items'] + out['LAD']['items'] + out['LCX']['items']
    unmeas = []
    for v, labs in conc_abn.items():
        its = left_items if v == 'LEFT' else out[v]['items']
        if any(not _covered(its, l) for l in labs):
            unmeas.append(v)
    for v in VESSELS:
        res[f'{v}_conc_unmeas'] = (v in unmeas) or (v != 'RCA' and 'LEFT' in unmeas)
    res['conc_abn_vessels'] = ','.join(sorted(conc_abn))
    res['conc_unmeas'] = bool(unmeas)
    res['table_ref'] = bool(re.search(r'测值见表|见表|如表|表格|见附表', f))
    return res


t0 = time.time()
MX = pd.DataFrame([extract_measurements(f, c) for f, c in zip(e['儿童超声心动图所见'].tolist(), e['儿童超声心动图结论'].tolist())], index=e.index)
e = pd.concat([e, MX], axis=1)
print(f'extraction {time.time()-t0:.0f} s | reports with any coronary diameter:',
      int(sum((e[f'{v}_items'].map(len) > 0) for v in VESSELS).gt(0).sum()),
      '| with a written Z:', int(sum((e[f'{v}_Zrep'].notna()) for v in VESSELS).gt(0).sum()),
      '| with height and weight:', int((e['height_cm'].notna() & e['weight_kg'].notna()).sum()))

## 5. Growth records and body-surface area
Height/weight from the paediatric admission records (dated by the admission) and from echo reports; records outside plausible ranges
(height 45–200 cm, weight 2–150 kg, BMI 8–40 kg/m², age ≥ 0) are dropped. BSA by Stevenson (0.0061·H + 0.0128·W − 0.1529), the formula the
laboratory's printed Z-scores are consistent with (Section 6). The cohort's own age–sex median BSA (0.25-y bins < 3 y, 0.5-y bins
thereafter) is used to project an older record to the echo date and, as a last resort, directly.

In [ ]:
def ste(H, W):
    return 0.0061 * H + 0.0128 * W - 0.1529


ar = T['儿科入院记录'][['科研患者编号', '科研就诊编号', '体重(kg)', '身高(cm)']].copy()
ar['W'] = pd.to_numeric(ar['体重(kg)'], errors='coerce'); ar['H'] = pd.to_numeric(ar['身高(cm)'], errors='coerce')
ar = ar.merge(adm[['科研就诊编号', '入院日期']], on='科研就诊编号', how='left')
ar['t'] = ar['入院日期'].dt.normalize(); ar['src'] = 'admission'
er = e.loc[e['height_cm'].notna() & e['weight_kg'].notna(), ['科研患者编号', '检查时间', 'height_cm', 'weight_kg']].rename(columns={'height_cm': 'H', 'weight_kg': 'W'})
er['t'] = er['检查时间'].dt.normalize(); er['src'] = 'echo_text'
GR = pd.concat([ar[['科研患者编号', 't', 'H', 'W', 'src']], er[['科研患者编号', 't', 'H', 'W', 'src']]]).dropna()
GR = GR.merge(PT, on='科研患者编号', how='inner')
GR['age_y'] = (GR['t'] - GR['dob']).dt.days / 365.25
GR['BMI'] = GR['W'] / (GR['H'] / 100) ** 2
n_raw = len(GR)
GR = GR[GR['H'].between(45, 200) & GR['W'].between(2, 150) & GR['BMI'].between(8, 40) & (GR['age_y'] >= 0)].drop_duplicates(['科研患者编号', 't', 'H', 'W'])
GR['BSA'] = ste(GR['H'], GR['W'])
print('growth records:', n_raw, '→ plausible and de-duplicated', len(GR), GR['src'].value_counts().to_dict(), '| children', GR['科研患者编号'].nunique())
bins = np.r_[np.arange(0, 3, 0.25), np.arange(3, 18.01, 0.5)]
GR['abin'] = pd.cut(GR['age_y'], bins, right=False)
REF = GR.dropna(subset=['abin']).groupby(['性别', 'abin'], observed=True)['BSA'].agg(['median', 'size']).reset_index()
REF['age_mid'] = REF['abin'].map(lambda iv: (iv.left + iv.right) / 2).astype(float)
REF = REF.rename(columns={'median': 'ref_bsa', 'size': 'n'})[['性别', 'age_mid', 'ref_bsa', 'n']]
GR = GR.drop(columns=['abin'])


def ref_bsa(sex, age):
    r = REF[REF['性别'] == sex]
    if len(r) == 0 or not np.isfinite(age):
        return np.nan
    return float(np.interp(age, r['age_mid'].values, r['ref_bsa'].values))


REF.to_csv(os.path.join(OUT_DIR, 'TableS1_reference_BSA_by_age_sex.csv'), index=False)

## 6. Reconstruction of the laboratory's Z-score equations
Pairs = (diameter, the Z written right after it) in reports that also print height and weight. For each vessel, Z = (D − μ(BSA)) / σ with
μ = a + b·√BSA + c·BSA and constant σ, fitted by robust least squares (soft-L1). Agreement with the written Z is the test.

In [ ]:
pr = []
sub = e[e['height_cm'].notna() & e['weight_kg'].notna()]
for v in ['LMCA', 'LAD', 'RCA']:
    for i, H, W, yr, it in zip(sub.index, sub['height_cm'], sub['weight_kg'], sub['yr'], sub[f'{v}_items']):
        for d, zv, seg in it:
            if np.isfinite(zv) and not (d >= 10 and float(d).is_integer()):
                pr.append(dict(vessel=v, D=d, Z=zv, seg=seg, H=H, W=W, yr=yr, rid=i))
PAIRS = pd.DataFrame(pr); PAIRS['BSA'] = ste(PAIRS['H'], PAIRS['W'])
EQ = {}; fits = []
for v in ['LMCA', 'LAD', 'RCA']:
    s = PAIRS[(PAIRS['vessel'] == v) & (PAIRS['Z'].abs() < 30)]
    B, Dm, Zm = s['BSA'].values, s['D'].values, s['Z'].values
    r = least_squares(lambda p: (Dm - (p[0] + p[1] * np.sqrt(B) + p[2] * B)) / p[3] - Zm, [0.5, 2.5, 0.0, 0.3], loss='soft_l1', f_scale=0.05, max_nfev=50000)
    EQ[v] = r.x; res = (Dm - (r.x[0] + r.x[1] * np.sqrt(B) + r.x[2] * B)) / r.x[3] - Zm
    fits.append(dict(vessel=v, n_pairs=len(s), n_reports=s['rid'].nunique(), a=r.x[0], b_sqrtBSA=r.x[1], c_BSA=r.x[2], SD_mm=r.x[3],
                     median_abs_error=np.median(np.abs(res)), pct_within_0_1=np.mean(np.abs(res) < 0.1) * 100,
                     pct_within_0_5=np.mean(np.abs(res) < 0.5) * 100, years=f"{int(s['yr'].min())}–{int(s['yr'].max())}"))
FITS = pd.DataFrame(fits)
FITS.round(4).to_csv(os.path.join(OUT_DIR, 'Table1_reconstructed_Z_equations.csv'), index=False)
print(FITS.round(4).to_string(index=False))


def z_from_d(v, D, B):
    a, b, c, d = EQ[v]
    return (D - (a + b * np.sqrt(B) + c * B)) / d


PAIRS['Zc'] = [z_from_d(v, D, B) for v, D, B in zip(PAIRS['vessel'], PAIRS['D'], PAIRS['BSA'])]
PAIRS['err'] = PAIRS['Zc'] - PAIRS['Z']
PAIRS['bsa_bin'] = pd.cut(PAIRS['BSA'], [0, 0.5, 0.8, 1.1, 1.4, 3])
eq_by_bsa = PAIRS.groupby('bsa_bin', observed=True)['err'].agg(n='size', median_abs=lambda s: s.abs().median(), pct_within_0_5=lambda s: (s.abs() < 0.5).mean() * 100)
eq_by_year = PAIRS.groupby('yr')['err'].agg(n='size', median_abs=lambda s: s.abs().median(), pct_within_0_5=lambda s: (s.abs() < 0.5).mean() * 100)
eq_by_bsa.round(3).to_csv(os.path.join(OUT_DIR, 'TableS2a_equation_error_by_BSA.csv')); eq_by_year.round(3).to_csv(os.path.join(OUT_DIR, 'TableS2b_equation_error_by_year.csv'))
KEY['equations'] = dict(n_pairs=int(len(PAIRS)), n_reports=int(PAIRS['rid'].nunique()),
                        median_abs_error=float(PAIRS['err'].abs().median()), pct_within_0_1=float((PAIRS['err'].abs() < 0.1).mean() * 100),
                        pct_within_0_5=float((PAIRS['err'].abs() < 0.5).mean() * 100),
                        pct_within_0_5_bsa_gt_1_4=float((PAIRS.loc[PAIRS['BSA'] > 1.4, 'err'].abs() < 0.5).mean() * 100))
print(eq_by_bsa.round(3)); print(KEY['equations'])

## 7. A Z-score for every written diameter
Written Z preferred for that diameter; otherwise computed (LMCA/LAD/RCA; the laboratory prints no LCX equation, so LCX counts only through a
written Z or the ≥ 8 mm rule). Integer diameters ≥ 10 mm are checked for a lost decimal point: with a written Z the reading that reproduces
it is kept; without one the value is dropped unless the vessel itself is called dilated/aneurysmal. BSA-source hierarchy as in Section 5;
the error of the age–sex fallback is measured on reports that print height and weight.

In [ ]:
e['t'] = e['检查时间'].dt.normalize()
e = e.merge(PT, on='科研患者编号', how='left')
e['age_y_echo'] = (e['t'] - e['dob']).dt.days / 365.25
e['BSA'] = np.where(e['height_cm'].notna() & e['weight_kg'].notna(), ste(e['height_cm'], e['weight_kg']), np.nan)
e['BSA_src'] = np.where(e['BSA'].notna(), 'same_report', '')
need = e['BSA'].isna()
g_ = GR[['科研患者编号', 't', 'BSA', 'age_y']].rename(columns={'t': 't_rec', 'BSA': 'BSA_rec', 'age_y': 'age_rec'}).sort_values('t_rec')
left = e.loc[need, ['科研患者编号', 't', 'age_y_echo', '性别']].reset_index().sort_values('t')
mm_ = pd.merge_asof(left, g_, left_on='t', right_on='t_rec', by='科研患者编号', direction='nearest')
within = (mm_['t'] - mm_['t_rec']).dt.days.abs() <= 90
proj = [b * ref_bsa(s, a1) / ref_bsa(s, a0) if np.isfinite(b) and np.isfinite(a0) and np.isfinite(a1) else np.nan
        for b, s, a0, a1 in zip(mm_['BSA_rec'], mm_['性别'], mm_['age_rec'], mm_['age_y_echo'])]
e.loc[mm_['index'].values, 'BSA'] = np.where(within, mm_['BSA_rec'], proj)
e.loc[mm_['index'].values, 'BSA_src'] = np.where(mm_['BSA_rec'].isna(), '', np.where(within, 'record_le90d', 'projected'))
last = e['BSA'].isna() & e['age_y_echo'].notna() & e['性别'].notna()
e.loc[last, 'BSA'] = [ref_bsa(s, a) for s, a in zip(e.loc[last, '性别'], e.loc[last, 'age_y_echo'])]
e.loc[last & e['BSA'].notna(), 'BSA_src'] = 'age_sex_reference'
e.loc[e['BSA'].isna(), 'BSA_src'] = 'none'
print('BSA source:', e['BSA_src'].value_counts().to_dict())
# error of the age-sex fallback, measured where the report prints height, weight and Z
chk = PAIRS.merge(e[['age_y_echo', '性别']], left_on='rid', right_index=True)
chk['Zref'] = [z_from_d(v, D, ref_bsa(s, a)) for v, D, s, a in zip(chk['vessel'], chk['D'], chk['性别'], chk['age_y_echo'])]
KEY['bsa_fallback'] = dict(n=int(chk['Zref'].notna().sum()), median_abs_error=float((chk['Zref'] - chk['Z']).abs().median()),
                           p90_abs_error=float((chk['Zref'] - chk['Z']).abs().quantile(0.9)),
                           aneurysm_agreement=float(((chk['Zref'] >= 2.5) == (chk['Z'] >= 2.5)).mean() * 100))
print('age-sex BSA fallback vs written Z:', KEY['bsa_fallback'])

ZV = ['LMCA', 'LAD', 'RCA']


def vessel_z(v, items, zfree, B, abn):
    """(max Z, source of the max, max diameter after the decimal check, n decimal fixes, n dropped)"""
    best, src, dmax, nfix, ndrop = -np.inf, '', -np.inf, 0, 0
    for D, Zp, seg in items:
        if D >= 10 and float(D).is_integer():
            if np.isfinite(Zp) and np.isfinite(B) and v in ZV:
                if abs(z_from_d(v, D / 10, B) - Zp) < abs(z_from_d(v, D, B) - Zp):
                    D = D / 10; nfix += 1
            elif not abn:
                ndrop += 1
                continue
        dmax = max(dmax, D)
        if np.isfinite(Zp):
            z, s = Zp, 'reported'
        elif np.isfinite(B) and v in ZV:
            z, s = z_from_d(v, D, max(B, 0.1)), 'computed'
        else:
            continue
        if z > best:
            best, src = z, s
    for z in zfree:
        if z > best:
            best, src = z, 'reported'
    return (best if np.isfinite(best) else np.nan), src, (dmax if np.isfinite(dmax) else np.nan), nfix, ndrop


conc = e['conc_abn_vessels'].fillna('').str.split(',')
for v in VESSELS:
    ca = conc.map(lambda l: v in l or (v != 'RCA' and 'LEFT' in l)).values
    res = [vessel_z(v, it, zf, B, bool(at or c)) for it, zf, B, at, c in zip(e[f'{v}_items'], e[f'{v}_Zfree'], e['BSA'], e[f'{v}_abn_text'], ca)]
    e[f'{v}_Zfinal'] = [r[0] for r in res]; e[f'{v}_Zsrc'] = [r[1] for r in res]; e[f'{v}_Dclean'] = [r[2] for r in res]
    e[f'{v}_nfix'] = [r[3] for r in res]; e[f'{v}_ndrop'] = [r[4] for r in res]
zc = e[[f'{v}_Zfinal' for v in VESSELS]]
e['maxZ'] = zc.max(axis=1)
has = zc.notna().any(axis=1)
e['maxZ_vessel'] = ''
e.loc[has, 'maxZ_vessel'] = zc[has].idxmax(axis=1).str.replace('_Zfinal', '', regex=False)
e['maxZ_src'] = ''
for v in VESSELS:
    sel = e['maxZ_vessel'] == v
    e.loc[sel, 'maxZ_src'] = e.loc[sel, f'{v}_Zsrc']
e['maxD'] = e[[f'{v}_Dclean' for v in VESSELS]].max(axis=1)
flag = {v: e[f'{v}_abn_unmeas'].astype(bool) | e[f'{v}_conc_unmeas'].astype(bool) for v in VESSELS}
left_ok = (e[['LMCA_Zfinal', 'LAD_Zfinal', 'LCX_Zfinal']].max(axis=1) >= 2.5) | (e[['LMCA_Dclean', 'LAD_Dclean', 'LCX_Dclean']].max(axis=1) >= 8)
right_ok = (e['RCA_Zfinal'] >= 2.5) | (e['RCA_Dclean'] >= 8)
e['unmeas_left'] = (flag['LMCA'] | flag['LAD'] | flag['LCX']) & ~left_ok
e['unmeas_right'] = flag['RCA'] & ~right_ok
e['abn_unmeas_any'] = e['unmeas_left'] | e['unmeas_right']


def aha(Z, Dmax):
    if np.isfinite(Dmax) and Dmax >= 8:
        return 4
    if not np.isfinite(Z):
        return np.nan
    return 4 if Z >= 10 else 3 if Z >= 5 else 2 if Z >= 2.5 else 1 if Z >= 2 else 0


e['aha_num'] = [aha(a, b) for a, b in zip(e['maxZ'], e['maxD'])]
e['num_status'] = np.where(e['aha_num'].isna(), 'none', np.where(e['abn_unmeas_any'], 'lower_bound', 'complete'))
KEY['numeric'] = dict(reports=int(len(e)), with_numeric=int(e['aha_num'].notna().sum()), complete=int((e['num_status'] == 'complete').sum()),
                      lower_bound=int((e['num_status'] == 'lower_bound').sum()), decimal_fixes=int(e[[f'{v}_nfix' for v in VESSELS]].sum().sum()),
                      integer_values_dropped=int(e[[f'{v}_ndrop' for v in VESSELS]].sum().sum()),
                      max_from_written_Z=int((e['maxZ_src'] == 'reported').sum()), max_from_computed_Z=int((e['maxZ_src'] == 'computed').sum()))
print(KEY['numeric'])

## 8. Tier 3 – measurements transcribed into clinical notes
Discharge summaries (诊疗经过) and outpatient notes (辅检结果, 现病史) often copy the echo numbers, including those that the report itself
only puts in the unexported table ("测值见表"). A note segment is used when it carries an echo keyword and a coronary diameter or Z and
does not name another hospital. Linking: a segment with a full calendar date → the report of the same child on that date; an undated
outpatient 辅检结果 → the child's report on the visit date or up to 7 days before. Linkage accuracy is measured on reports that have
their own numbers.

In [ ]:
DATE = re.compile(r'(20[12]\d)\s*[-./年]\s*(\d{1,2})\s*[-./月]\s*(\d{1,2})\s*日?')
ECHO_KW = re.compile(r'心脏彩超|超声心动图|心超|冠脉超声|心脏超声|彩超')
OTHER_HOSP = re.compile(r'医院|保健院|卫生院|诊所|外院|当地')
COR_NUM = re.compile(r'(?:LCA|LAD|RCA|左冠|右冠|前降|冠状动脉|冠脉)[^。；;\n]{0,40}?\d+(?:\.\d+)?\s*(?:mm|MM)|[Zz]\s*值')


def note_segments(text):
    if not isinstance(text, str) or not COR_NUM.search(text):
        return []
    ms = list(DATE.finditer(text)); out = []
    if not ms:
        return [(None, text)]
    if ms[0].start() > 0:
        out.append((None, text[:ms[0].start()]))
    for i, m_ in enumerate(ms):
        end = ms[i + 1].start() if i + 1 < len(ms) else len(text)
        try:
            dt = pd.Timestamp(int(m_.group(1)), int(m_.group(2)), int(m_.group(3)))
        except ValueError:
            dt = None
        out.append((dt, text[m_.end():end]))
    return out


def n_numbers(r):
    return sum(len(r[f'{v}_items']) + len(r[f'{v}_Zfree']) for v in VESSELS)


H = []
dsum = T['住院病历出院记录']
for pid, enc, t_ in zip(dsum['科研患者编号'], dsum['科研就诊编号'], txt(dsum['诊疗经过'])):
    for dt, seg in note_segments(t_):
        if dt is None or not (ECHO_KW.search(seg[:40]) or re.search(r'冠状动脉起源|冠脉起源|LCA|RCA', seg)) or OTHER_HOSP.search(seg[:40]):
            continue
        r = extract_measurements(seg, '')
        if n_numbers(r):
            H.append(dict(pid=pid, src='discharge_summary', cited=dt, visit=pd.NaT, dated=True, res=r))
op = T['门急诊病历'].copy(); op['vd'] = pd.to_datetime(op['门诊就诊日期'], errors='coerce').dt.normalize()
for col in ['辅检结果', '门诊病历现病史']:
    for pid, vd, t_ in zip(op['科研患者编号'], op['vd'], txt(op[col])):
        for dt, seg in note_segments(t_):
            if dt is None and col != '辅检结果':
                continue
            if dt is not None and not (ECHO_KW.search(seg[:40]) or re.search(r'冠状动脉起源|冠脉起源|LCA|RCA', seg)):
                continue
            if OTHER_HOSP.search(seg[:40]) or not COR_NUM.search(seg):
                continue
            r = extract_measurements(seg, '')
            if n_numbers(r):
                H.append(dict(pid=pid, src='outpatient_' + col, cited=dt, visit=vd, dated=dt is not None, res=r))
H = pd.DataFrame(H)
print('note segments with coronary numbers:', len(H), H.groupby(['src', 'dated']).size().to_dict())

E = e[['科研患者编号', 't', 'BSA', 'cor_status']].reset_index().rename(columns={'index': 'rid'}).sort_values('t')
Eg = {k: g for k, g in E.groupby('科研患者编号')}
links = []
for i, r in H.iterrows():
    zp = Eg.get(r['pid'])
    if zp is None:
        continue
    if r['dated']:
        m_ = zp[zp['t'] == r['cited']]; how = 'dated_exact'
    else:
        m_ = zp[(zp['t'] <= r['visit']) & (zp['t'] >= r['visit'] - pd.Timedelta(days=7))]; m_ = m_.iloc[[-1]] if len(m_) else m_; how = 'undated_le7d'
    if len(m_) == 0:
        continue
    rid, B, ws = int(m_['rid'].iloc[0]), m_['BSA'].iloc[0], m_['cor_status'].iloc[0]
    res = r['res']; vz = {}; dmax = np.nan
    for v in VESSELS:
        zv, s_, dm, _, _ = vessel_z(v, res[f'{v}_items'], res[f'{v}_Zfree'], B, res[f'{v}_abn_text'] or (ws >= 1))
        vz[v] = zv; dmax = np.nanmax([dmax, dm]) if np.isfinite(dm) else dmax
    zmax = np.nanmax(list(vz.values())) if any(np.isfinite(x) for x in vz.values()) else np.nan
    zl = np.nanmax([vz['LMCA'], vz['LAD'], vz['LCX'], -np.inf]); zr = vz['RCA'] if np.isfinite(vz['RCA']) else -np.inf
    big = np.isfinite(dmax) and dmax >= 8
    fl = any(res[f'{v}_abn_unmeas'] or res[f'{v}_conc_unmeas'] for v in ('LMCA', 'LAD', 'LCX')); fr = res['RCA_abn_unmeas'] or res['RCA_conc_unmeas']
    unmeas = (fl and zl < 2.5 and not big) or (fr and zr < 2.5 and not big)
    links.append(dict(rid=rid, how=how, src=r['src'], h_maxZ=zmax, h_maxD=dmax, h_aha=aha(zmax, dmax), h_complete=not unmeas))
L = pd.DataFrame(links)
L = L[L['h_aha'].notna()]
val = L.merge(e[['maxZ', 'aha_num', 'num_status']], left_on='rid', right_index=True)
val = val[(val['num_status'] == 'complete') & val['h_complete']]
link_acc = val.groupby('how').apply(lambda g: pd.Series(dict(n=len(g), exact_AHA_agreement=(g['h_aha'] == g['aha_num']).mean() * 100,
                                                            aneurysm_agreement=((g['h_aha'] >= 2) == (g['aha_num'] >= 2)).mean() * 100,
                                                            median_abs_Z_diff=(g['h_maxZ'] - g['maxZ']).abs().median())))
print(link_acc.round(2)); link_acc.round(3).to_csv(os.path.join(OUT_DIR, 'TableS3_note_linkage_accuracy.csv'))
KEY['tier3'] = dict(segments=int(len(H)), linked=int(len(L)), validation=link_acc.round(2).reset_index().to_dict('records'))
Lb = L.sort_values(['rid', 'h_complete', 'h_aha'], ascending=[True, False, False]).drop_duplicates('rid').set_index('rid')
e['h_aha'] = Lb['h_aha'].reindex(e.index); e['h_complete'] = Lb['h_complete'].reindex(e.index); e['h_how'] = Lb['how'].reindex(e.index)

## 9. Tier 4 – calibrated wording model
Text features of the coronary statements only (conclusion and findings separately): wording category, upper-normal wording, modifiers
(稍/轻度/临界, 明显/显著), aneurysm words, explicit size words (小型/中型/大型/巨大), beading, thrombus, calcification, wall changes, extent,
shape, change words (减小/增大/相仿), vessels involved, number of abnormal clauses, table reference, era. Model: multinomial logistic
regression (L2, C = 1) fitted by L-BFGS; trained on reports whose numeric category is complete. Validation: 5-fold cross-validation grouped
by child, temporal validation (train ≤ 2022, test ≥ 2023), and reports without complete numbers whose category was recovered from notes.

In [ ]:
NEG_T = re.compile(r'未见|未探及|无明显|不宽|不扩张|未扩张|无扩张|不增宽|未增宽|正常范围|内径正常|起源正常|未及|恢复正常|消失')
TF = {'mild': r'稍|略|轻度|轻微|偏宽|临界|较同龄', 'marked': r'明显|显著', 'aneur_word': r'瘤样|瘤状|冠脉瘤|动脉瘤|瘤形成|呈瘤',
      'size_giant': r'巨大|巨型|大型', 'size_medium': r'中型|中等大小', 'size_small': r'小型', 'beaded': r'串珠', 'thrombus': r'血栓|附壁|回声团|光团',
      'calcif': r'钙化|强回声|光斑', 'wall': r'毛糙|增厚|回声增强|内膜|管壁', 'extent': r'全程|弥漫|多发|多处|两处|多个', 'saccular': r'囊状|囊样|球状|球形',
      'fusiform': r'梭形|梭状', 'regress': r'减小|缩小|好转|回缩|变窄|较前.{0,4}(减|缩|小)', 'progress': r'增大|进展|较前.{0,4}(增|扩|大)',
      'stable': r'变化不著|相仿|无明显变化|大致同前|同前'}
TF = {k: re.compile(v) for k, v in TF.items()}
TV = {'LMCA': re.compile(r'主干|LCA|LMCA'), 'LAD': re.compile(r'前降|LAD'), 'LCX': re.compile(r'回旋|旋支|LCX', re.I), 'RCA': re.compile(r'右冠|RCA'),
      'LEFT': re.compile(r'左冠|左、右冠|左右冠|双侧冠')}


def text_features(findings, conclusion):
    fc, cc = wclean(findings), wclean(conclusion); out = {}
    for tag, tx in (('c', cc), ('f', fc)):
        st, up = status_from_text(tx)
        out[f'w_{tag}'] = -1 if (isinstance(st, float) and np.isnan(st)) else int(st)
        out[f'upper_{tag}'] = int(bool(up))
        abn_cl = [c for l in coronary_lines(tx) for c in re.split(r'[，,]', l)
                  if (ANEU_W.search(c) or DIL_W.search(c) or TF['beaded'].search(c) or TF['thrombus'].search(c)) and not NEG_T.search(c)]
        j = '，'.join(abn_cl)
        for k, p in TF.items():
            out[f'{k}_{tag}'] = int(bool(p.search(j)))
        ves = {k: int(bool(p.search(j))) for k, p in TV.items()}
        if ves['LEFT'] and re.search(r'右冠|左、右|左右|双侧', j):
            ves['RCA'] = 1
        for k, v in ves.items():
            out[f'v_{k}_{tag}'] = v
        out[f'n_abn_clauses_{tag}'] = min(len(abn_cl), 6)
    out['table_ref_t'] = int(bool(re.search(r'测值见表|见表|如表|表格|见附表', fc)))
    return out


XT = pd.DataFrame([text_features(f, c) for f, c in zip(e['儿童超声心动图所见'].tolist(), e['儿童超声心动图结论'].tolist())], index=e.index)


def design(idx_):
    M = XT.loc[idx_].astype(float).copy()
    for tag in ('c', 'f'):
        for k in (-1, 0, 1, 2):
            M[f'w_{tag}_{k}'] = (XT.loc[idx_, f'w_{tag}'] == k).astype(float)
        M = M.drop(columns=[f'w_{tag}'])
    yr = e.loc[idx_, 'yr'].clip(2018, 2026)
    for lo, hi in ((2018, 2019), (2020, 2021), (2022, 2024), (2025, 2026)):
        M[f'era_{lo}_{hi}'] = ((yr >= lo) & (yr <= hi)).astype(float)
    return M.values


def fit_softmax(X, y, K=5, C=1.0):
    n, p = X.shape; Y = np.eye(K)[y]

    def f(w):
        W = w.reshape(p + 1, K); Zl = X @ W[1:] + W[0]
        Zl -= Zl.max(1, keepdims=True); P = np.exp(Zl); P /= P.sum(1, keepdims=True)
        loss = -np.sum(Y * np.log(P + 1e-300)) + 0.5 / C * np.sum(W[1:] ** 2)
        G = P - Y; g = np.vstack([G.sum(0), X.T @ G + W[1:] / C])
        return loss, g.ravel()
    r = minimize(f, np.zeros((p + 1) * K), jac=True, method='L-BFGS-B', options={'maxiter': 5000})
    return r.x.reshape(p + 1, K)


def predict_softmax(W, X):
    Zl = X @ W[1:] + W[0]; Zl -= Zl.max(1, keepdims=True); P = np.exp(Zl)
    return P / P.sum(1, keepdims=True)


def auc(y, s):
    y = np.asarray(y, bool); s = np.asarray(s, float)
    if y.all() or (~y).all():
        return np.nan
    r = stats.rankdata(s); n1 = y.sum(); n0 = (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def kappa_linear(a, b, K=5):
    O = np.zeros((K, K))
    for i, j in zip(a, b):
        O[i, j] += 1
    Wt = 1 - np.abs(np.subtract.outer(np.arange(K), np.arange(K))) / (K - 1)
    E_ = np.outer(O.sum(1), O.sum(0)) / O.sum()
    return 1 - ((1 - Wt) * O).sum() / ((1 - Wt) * E_).sum()


def metrics(y, P, tag):
    y = np.asarray(y, int); yhat = P.argmax(1)
    return dict(set=tag, n=len(y), exact_accuracy=np.mean(yhat == y) * 100, weighted_kappa=kappa_linear(y, yhat),
                auc_aneurysm=auc(y >= 2, P[:, 2:].sum(1)), auc_large=auc(y == 4, P[:, 4]),
                accuracy_aneurysm_yes_no=np.mean((P[:, 2:].sum(1) >= 0.5) == (y >= 2)) * 100,
                observed_aneurysm_pct=np.mean(y >= 2) * 100, predicted_aneurysm_pct=P[:, 2:].sum(1).mean() * 100,
                observed_large_pct=np.mean(y == 4) * 100, predicted_large_pct=P[:, 4].mean() * 100)


trn = e.index[(e['num_status'] == 'complete') & e['cor_status'].notna()]
Xtr, ytr = design(trn), e.loc[trn, 'aha_num'].astype(int).values
groups = e.loc[trn, '科研患者编号'].values
ug = np.unique(groups); rng = np.random.default_rng(2026); rng.shuffle(ug)
fold_of = {g: i % 5 for i, g in enumerate(ug)}; folds = np.array([fold_of[g] for g in groups])
Pcv = np.zeros((len(trn), 5))
for k in range(5):
    Wk = fit_softmax(Xtr[folds != k], ytr[folds != k]); Pcv[folds == k] = predict_softmax(Wk, Xtr[folds == k])
abn_tr = (e.loc[trn, 'cor_status'] >= 1).values
yrs = e.loc[trn, 'yr'].values
Wt_ = fit_softmax(Xtr[yrs <= 2022], ytr[yrs <= 2022]); Ptemp = predict_softmax(Wt_, Xtr[yrs >= 2023])
# wording-category-only benchmark
Xw = np.eye(3)[e.loc[trn, 'cor_status'].astype(int).values]
Pw = np.zeros((len(trn), 5))
for k in range(5):
    Wk = fit_softmax(Xw[folds != k], ytr[folds != k]); Pw[folds == k] = predict_softmax(Wk, Xw[folds == k])
WM = fit_softmax(Xtr, ytr)                                   # final model
MV = [metrics(ytr, Pcv, 'CV (grouped by child), all reports'), metrics(ytr[abn_tr], Pcv[abn_tr], 'CV, reports worded dilation/aneurysm'),
      metrics(ytr[~abn_tr], Pcv[~abn_tr], 'CV, reports worded normal'), metrics(ytr, Pw, 'CV, wording category alone (benchmark)'),
      metrics(ytr[yrs >= 2023], Ptemp, 'temporal: train <=2022, test >=2023')]
print('training reports:', len(trn))
print(pd.DataFrame(MV).round(3).to_string(index=False))
CM = pd.crosstab(pd.Series(ytr[abn_tr], name='AHA from numbers'), pd.Series(Pcv[abn_tr].argmax(1), name='model (CV) most likely'))
print(CM)

## 10. Final per-report category and per-child windows
Tier 3 values are used only when they are plausible for the report's own wording (probability ≥ 0.01 under the wording model); a note
that contradicts its report (e.g. a giant aneurysm transcribed for a report worded normal) is not used.

In [ ]:
Pm = predict_softmax(WM, design(e.index))
P = np.zeros((len(e), 5)); tier = np.zeros(len(e), int); floor = np.full(len(e), np.nan); n_conflict = 0
num, st_, src = e['aha_num'].values, e['num_status'].values, e['maxZ_src'].values
hh, hc = e['h_aha'].values.astype(float), e['h_complete'].fillna(False).values.astype(bool)
has_w = e['cor_status'].notna().values
for i in range(len(e)):
    if st_[i] == 'complete':
        P[i, int(num[i])] = 1; tier[i] = 1 if src[i] == 'reported' else 2; continue
    f_ = num[i] if st_[i] == 'lower_bound' else np.nan
    if np.isfinite(hh[i]):
        if has_w[i] and Pm[i, int(hh[i])] < 0.01:
            n_conflict += 1                                  # note contradicts the report's own wording: not used
        elif hc[i] and (not np.isfinite(f_) or hh[i] >= f_):
            P[i, int(hh[i])] = 1; tier[i] = 3; continue
        else:
            f_ = np.nanmax([f_, hh[i]])
    if not has_w[i] and not np.isfinite(f_):
        continue                                             # no coronary statement and no number
    p_ = Pm[i].copy()
    if np.isfinite(f_) and f_ > 0:
        p_[:int(f_)] = 0; p_ = p_ / p_.sum() if p_.sum() > 0 else np.eye(5)[int(f_)]
    P[i] = p_; tier[i] = 4; floor[i] = f_
e['tier'] = tier; e['floor'] = floor
for k in range(5):
    e[f'P{k}'] = P[:, k]
cdf = P.cumsum(1)
e['aha_best'] = np.where(tier > 0, (cdf >= 0.5).argmax(1), np.nan)
e['p_aneurysm'] = np.where(tier > 0, P[:, 2:].sum(1), np.nan); e['p_large'] = np.where(tier > 0, P[:, 4], np.nan)
KEY['tiers'] = {int(k): int(v) for k, v in e['tier'].value_counts().sort_index().items()}
KEY['tier3_conflicts_not_used'] = n_conflict
print('reports by evidence tier (0 = unclassifiable):', KEY['tiers'], '| note values contradicting the report wording (P < 0.01 under the wording model), not used:', n_conflict)

WIN = {'inhosp': e['phase'].isin(['pre_index', 'in_index']), 'le30': e['phase'].isin(['pre_index', 'in_index', 'post30']),
       'ever': e['phase'].isin(['pre_index', 'in_index', 'post30', 'd31_90', 'd91_365', 'gt365'])}


def child_window(mask, label):
    q = e[mask & (e['tier'] > 0)]
    C_ = q[[f'P{k}' for k in range(5)]].cumsum(axis=1).values
    q = q.assign(**{f'C{k}': C_[:, k] for k in range(5)}, exact=(q['tier'] <= 3))
    g = q.groupby('科研患者编号').agg(**{f'C{k}': (f'C{k}', 'min') for k in range(5)}, has_numeric=('exact', 'max'), n_reports=('tier', 'size'))
    g = g.join(q[q['exact']].groupby('科研患者编号')['aha_best'].max().rename('num_max'))
    Cm = g[[f'C{k}' for k in range(5)]].values
    out = pd.DataFrame(index=g.index)
    out[f'{label}_best'] = (Cm >= 0.5).argmax(1)
    out[f'{label}_p_aneurysm'] = 1 - Cm[:, 1]; out[f'{label}_p_large'] = 1 - Cm[:, 3]
    p_above = np.array([1 - Cm[i, int(n)] if np.isfinite(n) else np.nan for i, n in enumerate(g['num_max'].values)])
    out[f'{label}_evidence'] = np.where(~g['has_numeric'].astype(bool), 'wording_model_only', np.where(p_above < 0.1, 'numeric', 'numeric_plus_model'))
    out[f'{label}_n_reports'] = g['n_reports']
    return out


# Sensitivity of Tier 4 to the model specification: in 2025-2026 the reports that still carry numbers in the text are a selected
# (often more severe) minority, so a 2025-2026 era term may not transfer to the reports whose numbers sit in the unexported table.
# Alternative specification: era terms 2018-19 / 2020-21 / 2022 onward and no table-reference feature.
def design_alt(idx_):
    M = XT.loc[idx_].astype(float).drop(columns=['table_ref_t']).copy()
    for tag in ('c', 'f'):
        for k in (-1, 0, 1, 2):
            M[f'w_{tag}_{k}'] = (XT.loc[idx_, f'w_{tag}'] == k).astype(float)
        M = M.drop(columns=[f'w_{tag}'])
    yr = e.loc[idx_, 'yr'].clip(2018, 2026)
    for lo, hi in ((2018, 2019), (2020, 2021), (2022, 2026)):
        M[f'era_{lo}_{hi}'] = ((yr >= lo) & (yr <= hi)).astype(float)
    return M.values


WA = fit_softmax(design_alt(trn), ytr)
sens = []
for w_, lab in ((1, 'dilation'), (2, 'aneurysm')):
    qi = e.index[(e['yr'] >= 2025) & (e['cor_status'] == w_) & (e['tier'] == 4) & e['floor'].isna()]
    if len(qi) == 0:
        continue
    Pa = predict_softmax(WA, design_alt(qi)); Pb = e.loc[qi, [f'P{k}' for k in range(5)]].values
    for nm, P_ in (('primary', Pb), ('alternative', Pa)):
        sens.append(dict(wording=lab, model=nm, n=len(qi), mean_P_aneurysm=P_[:, 2:].sum(1).mean(), mean_P_medium_plus=P_[:, 3:].sum(1).mean(), mean_P_large=P_[:, 4].mean()))
SENS = pd.DataFrame(sens); SENS.round(3).to_csv(os.path.join(OUT_DIR, 'TableS8_tier4_specification_sensitivity_2025_2026.csv'), index=False)
KEY['tier4_sensitivity_2025_2026'] = SENS.round(3).to_dict('records')
print(SENS.round(3).to_string(index=False))

PAT = None
for k_, m_ in WIN.items():
    w_ = child_window(m_, k_)
    PAT = w_ if PAT is None else PAT.join(w_, how='outer')
for k_, m_ in WIN.items():
    PAT = PAT.join(e[m_].groupby('科研患者编号')['cor_status'].max().rename(f'{k_}_wording'))
PAT = PAT.join(idx.set_index('科研患者编号')[['index_enc', 'idx_adm', 'idx_dis', 'year']])
print('children with ≥1 classifiable report after the index admission:', len(PAT))
for k_ in WIN:
    print(k_, PAT[f'{k_}_best'].value_counts().sort_index().to_dict(), PAT[f'{k_}_evidence'].value_counts().to_dict())

## 11. What the words mean in Z-score terms (report level)
Reports with a complete numeric category (Tier 1–2). The subset whose maximal Z was written by the sonographer (no computation involved)
is shown separately, as is the share of "dilation"-worded reports that are AHA aneurysms by calendar year.

In [ ]:
cmp_ = e[(e['num_status'] == 'complete') & e['cor_status'].notna()].copy()
cmp_['wording'] = cmp_['cor_status'].map(WORD)
cmp_.loc[(cmp_['cor_status'] == 0) & cmp_['upper_normal'], 'wording'] = 'upper-normal'


def wtab(q):
    t = pd.crosstab(q['wording'], q['aha_num'].astype(int)).reindex(index=['normal', 'upper-normal', 'dilation', 'aneurysm'], columns=range(5), fill_value=0)
    pct = t.div(t.sum(1), axis=0) * 100
    return t, pct


t_all, p_all = wtab(cmp_); t_rep, p_rep = wtab(cmp_[cmp_['maxZ_src'] == 'reported'])
T2 = pd.concat({'n_all': t_all, 'pct_all': p_all.round(1), 'n_writtenZ': t_rep, 'pct_writtenZ': p_rep.round(1)}, axis=1)
T2.columns = [f'{a}_{CAT_SHORT[b]}' for a, b in T2.columns]
T2.to_csv(os.path.join(OUT_DIR, 'Table2_wording_vs_AHA_reports.csv'))
print(T2.to_string())
dil_year = cmp_[cmp_['cor_status'] == 1].groupby('yr').agg(n=('aha_num', 'size'), pct_AHA_aneurysm=('aha_num', lambda s: (s >= 2).mean() * 100),
                                                          pct_large=('aha_num', lambda s: (s == 4).mean() * 100))
dil_year = dil_year[dil_year['n'] >= 20]
dil_year.round(1).to_csv(os.path.join(OUT_DIR, 'TableS4_dilation_wording_by_year.csv')); print(dil_year.round(1).T)
# Z distribution by wording (numbers for the text)
zdesc = cmp_.groupby('wording')['maxZ'].describe(percentiles=[.25, .5, .75]).reindex(['normal', 'upper-normal', 'dilation', 'aneurysm'])
KEY['wording_vs_aha'] = {w: dict(n=int(t_all.loc[w].sum()), pct=[round(x, 1) for x in p_all.loc[w].tolist()],
                                 n_writtenZ=int(t_rep.loc[w].sum()), pct_writtenZ=[round(x, 1) for x in p_rep.loc[w].tolist()],
                                 median_Z=round(float(zdesc.loc[w, '50%']), 2), IQR=[round(float(zdesc.loc[w, '25%']), 2), round(float(zdesc.loc[w, '75%']), 2)])
                         for w in ['normal', 'upper-normal', 'dilation', 'aneurysm']}
KEY['dilation_wording_aneurysm_by_year'] = dil_year['pct_AHA_aneurysm'].round(1).to_dict()
for w, d_ in KEY['wording_vs_aha'].items():
    print(w, d_)

## 12. External anchors: treatment and discharge coding (child level)
AHA 2017 recommends anticoagulation (warfarin or low-molecular-weight heparin) for large/giant aneurysms and considers dual antiplatelet
therapy for medium ones. Warfarin (inpatient orders or outpatient prescriptions/notes) is therefore a specific marker of large/giant
aneurysms; clopidogrel of medium-or-larger. Low-molecular-weight heparin is also used for other indications and is shown for completeness.
Discharge coding (冠状动脉扩张 vs 冠状动脉动脉瘤) is compared with the in-hospital AHA category.

In [ ]:
mo = T['住院用药医嘱'][['科研患者编号', '住院用药医嘱药品名称', '住院用药医嘱用法']].copy()
nm_, use_ = mo['住院用药医嘱药品名称'].fillna(''), mo['住院用药医嘱用法'].fillna('')
lmwh_ids = set(mo.loc[nm_.str.contains('依诺肝素|低分子') & ~use_.str.contains('冲管|封管'), '科研患者编号'])
warf_ids = set(mo.loc[nm_.str.contains('华法林'), '科研患者编号']); clop_ids = set(mo.loc[nm_.str.contains('氯吡格雷|波立维'), '科研患者编号'])
optxt = txt(op['门诊病历治疗意见']) + ' ' + txt(op['门诊病历现病史'])
warf_ids |= set(op.loc[optxt.str.contains('华法林|华法令'), '科研患者编号']); clop_ids |= set(op.loc[optxt.str.contains('氯吡格雷|波立维'), '科研患者编号'])
PAT['warfarin'] = PAT.index.isin(warf_ids); PAT['clopidogrel'] = PAT.index.isin(clop_ids); PAT['lmwh'] = PAT.index.isin(lmwh_ids)
by_aha = PAT.groupby('ever_best').agg(n=('warfarin', 'size'), warfarin_pct=('warfarin', lambda s: s.mean() * 100), clopidogrel_pct=('clopidogrel', lambda s: s.mean() * 100),
                                      lmwh_pct=('lmwh', lambda s: s.mean() * 100))
by_aha.index = [CAT_SHORT[int(i)] for i in by_aha.index]
by_word = PAT.groupby(PAT['ever_wording'].map(WORD)).agg(n=('warfarin', 'size'), warfarin_pct=('warfarin', lambda s: s.mean() * 100),
                                                         clopidogrel_pct=('clopidogrel', lambda s: s.mean() * 100), lmwh_pct=('lmwh', lambda s: s.mean() * 100)).reindex(['normal', 'dilation', 'aneurysm'])
T3 = pd.concat({'AHA category (whole follow-up)': by_aha, 'report wording (whole follow-up)': by_word})
T3.round(1).to_csv(os.path.join(OUT_DIR, 'Table3_treatment_anchor.csv')); print(T3.round(1))
wf = PAT[PAT['warfarin']]
KEY['warfarin'] = dict(n=int(len(wf)), aha=wf['ever_best'].value_counts().sort_index().to_dict(), wording=wf['ever_wording'].map(WORD).value_counts().to_dict())
print('warfarin-treated children:', KEY['warfarin'])
# discharge coding at the index admission
dn = T['病案出院诊断归一']
dn = dn[dn['科研就诊编号'].isin(set(idx['index_enc']))]
code_an = set(dn.loc[dn['诊断疾病名称'].str.contains('冠状动脉动脉瘤|冠状动脉瘤'), '科研患者编号'])
code_dil = set(dn.loc[dn['诊断疾病名称'].str.contains('冠状动脉扩张'), '科研患者编号'])
PAT['index_coding'] = np.select([PAT.index.isin(code_an), PAT.index.isin(code_dil)], ['coded aneurysm', 'coded dilation'], 'no coronary code')
q_ = PAT[PAT['inhosp_best'].notna() & PAT['inhosp_evidence'].notna()]
T4 = pd.crosstab(q_['inhosp_best'].astype(int).map(dict(enumerate(CAT_SHORT))), q_['index_coding']).reindex(CAT_SHORT)
T4p = T4.div(T4.sum(1), axis=0) * 100
pd.concat({'n': T4, 'row_pct': T4p.round(1)}, axis=1).to_csv(os.path.join(OUT_DIR, 'Table4_discharge_coding_vs_AHA.csv'))
print(pd.concat({'n': T4, 'row %': T4p.round(1)}, axis=1))
KEY['coding'] = {c: dict(n=int(T4.loc[c].sum()), coded_aneurysm_pct=round(float(T4p.loc[c].get('coded aneurysm', 0)), 1),
                         coded_dilation_pct=round(float(T4p.loc[c].get('coded dilation', 0)), 1)) for c in CAT_SHORT}

## 13. Crosswalk to the frozen master v1.3 (reference only)
The master's `Coronary_Severity_3` (0 normal / 1 dilation only / 2 aneurysm) is compared with this notebook's in-hospital category collapsed
the same way (0 = Z < 2; 1 = 2–< 2.5; 2 = Z ≥ 2.5 or ≥ 8 mm), by the master's `severity_source`. Children whose master label is
"qualitative dilation" but whose in-hospital reports in this export are all worded normal are traced to outside-hospital echo text in the
admission narrative (an echo done before this admission, usually at another hospital) or to discharge coding.

In [ ]:
MASTER = os.path.join(DATA_DIR, 'KD_Infant_Master_v1.3.xlsx')
if os.path.exists(MASTER):
    fm = pd.read_excel(MASTER, sheet_name=0, usecols=['Patient_ID', 'Coronary_Severity_3', 'severity_source'])
    fm['科研患者编号'] = fm['Patient_ID'].astype(str)
    X_ = PAT.join(fm.set_index('科研患者编号'), how='inner')
    X_ = X_[X_['inhosp_best'].notna() & X_['Coronary_Severity_3'].notna()]
    X_['ours3'] = np.select([X_['inhosp_best'] >= 2, X_['inhosp_best'] == 1], [2, 1], 0)
    T5 = pd.crosstab([X_['severity_source'], X_['Coronary_Severity_3'].astype(int)], X_['ours3'], margins=True)
    T5.to_csv(os.path.join(OUT_DIR, 'Table5_master_v13_crosswalk.csv')); print(T5)
    qd = X_[X_['severity_source'] == 'qualitative_dilation']
    ids_nd = set(qd.index[qd['inhosp_wording'] == 0])
    ds_idx = T['住院病历出院记录'][T['住院病历出院记录']['科研就诊编号'].isin(set(idx['index_enc']))]
    # a non-negated coronary dilation statement in the admission narrative (入院情况) describes an echo done before this admission
    def pre_adm_dilation(t):
        for m_ in re.finditer(r'冠[^。；;，,]{0,12}(扩张|增宽|增粗)', t):
            if not re.search(r'未见|无|不|正常', t[max(0, m_.start() - 6):m_.end()]):
                return True
        return False
    outside = set(ds_idx.loc[txt(ds_idx['入院情况']).map(pre_adm_dilation), '科研患者编号'])
    KEY['master'] = dict(overlap=int(len(X_)), exact_agreement_pct=round(float((X_['ours3'] == X_['Coronary_Severity_3']).mean() * 100), 1),
                         qualitative_dilation=dict(n=int(len(qd)), ours=qd['ours3'].value_counts().sort_index().to_dict(),
                                                   all_inhospital_reports_normal=len(ids_nd), of_which_preadmission_echo_in_notes=len(ids_nd & outside),
                                                   of_which_coded_dilation=len(ids_nd & code_dil)),
                         master_class1=dict(n=int((X_['Coronary_Severity_3'] == 1).sum()), ours=X_.loc[X_['Coronary_Severity_3'] == 1, 'ours3'].value_counts().sort_index().to_dict()),
                         master_class2=dict(n=int((X_['Coronary_Severity_3'] == 2).sum()), ours=X_.loc[X_['Coronary_Severity_3'] == 2, 'ours3'].value_counts().sort_index().to_dict()))
    print(KEY['master'])
else:
    print('frozen master not present - crosswalk skipped')

## 14. Blinded manual extraction audits
Two stratified random samples of reports were read by independent readers blinded to the extractor output (`audit/`): a development
sample (n = 200, used to find and fix extraction errors) and an independent validation sample (n = 150, drawn afterwards from reports
not in the development sample). Agreement = same maximal diameter per vessel (±0.05 mm) and same maximal written Z per vessel (±0.01),
both missing counting as agreement; the "abnormal segment without its own diameter" flag is compared among reports with numbers.

In [ ]:
AUD = []
for name, key_f, parts in [('development', 'numeric_audit_key.csv', ['reader_part1.csv', 'reader_part2.csv']),
                           ('validation', 'validation_audit_key.csv', ['validation_reader_part1.csv', 'validation_reader_part2.csv']),
                           ('final_check', 'final_audit_key.csv', ['final_reader.csv'])]:
    kp = os.path.join(DATA_DIR, 'audit', key_f)
    if not (os.path.exists(kp) and all(os.path.exists(os.path.join(DATA_DIR, 'audit', p)) for p in parts)):
        continue
    K_ = pd.read_csv(kp); R_ = pd.concat([pd.read_csv(os.path.join(DATA_DIR, 'audit', p)) for p in parts])
    A_ = K_.merge(R_, on='audit_id').merge(e[['report_key'] + [f'{v}_Dclean' for v in VESSELS] + [f'{v}_Zrep' for v in ['LMCA', 'LAD', 'RCA']] +
                                             ['abn_unmeas_any', 'num_status']].drop_duplicates('report_key'), on='report_key', how='left')

    def ag(a, b, tol):
        a = pd.to_numeric(a, errors='coerce'); b = pd.to_numeric(b, errors='coerce')
        return (a.isna() & b.isna()) | ((a - b).abs() <= tol)
    allok = np.ones(len(A_), bool); row = dict(sample=name, n=len(A_))
    for v in VESSELS:
        ok = ag(A_[f'{v}_Dclean'], A_[f'{v}_D'], 0.05); allok &= ok.values; row[f'{v}_diameter_pct'] = ok.mean() * 100
    for v in ['LMCA', 'LAD', 'RCA']:
        ok = ag(A_[f'{v}_Zrep'], A_[f'{v}_Z'], 0.01); allok &= ok.values; row[f'{v}_Z_pct'] = ok.mean() * 100
    row['all_seven_fields'] = f'{allok.sum()}/{len(A_)}'; row['all_seven_pct'] = allok.mean() * 100
    q = A_[A_['num_status'] != 'none']
    xu, ru = q['abn_unmeas_any'].astype(bool), q['unmeasured_abnormal'].astype(str).str.strip().str.lower().eq('yes')
    row['unmeasured_flag_agree'] = f'{(xu == ru).sum()}/{len(q)}'; row['unmeasured_missed_by_extractor'] = int((~xu & ru).sum())
    row['unmeasured_overcalled_by_extractor'] = int((xu & ~ru).sum())
    AUD.append(row)
    A_.to_csv(os.path.join(OUT_DIR, f'TableS5_audit_{name}_merged.csv'), index=False)
AUD = pd.DataFrame(AUD)
logf = os.path.join(DATA_DIR, 'audit', 'audit_log.json')
if os.path.exists(logf):
    KEY['audit_log_first_look'] = json.load(open(logf, encoding='utf-8'))
    print('audit log (first look, before fixes prompted by each sample):', json.dumps(KEY['audit_log_first_look'], ensure_ascii=False, indent=1))
if len(AUD):
    AUD.round(1).to_csv(os.path.join(OUT_DIR, 'Table6_blinded_audits.csv'), index=False); print(AUD.round(1).T)
    KEY['audits'] = AUD.round(1).to_dict('records')
else:
    print('audit files not present - skipped')

## 15. Impact on the locked surveillance results (cohort C1 of the v3 notebook)
C1 = index KD admission 2018 onward, not 2021, with ≥ 1 in-hospital report with a coronary statement (definitions unchanged from v3).
Each v3 wording-based quantity is recomputed with the AHA category (best estimate per report; child level = maximum in the window).

In [ ]:
inh = e[(e['phase'] == 'in_index') & e['cor_status'].notna()]
C1 = set(inh.loc[(inh['idx_year'] >= 2018) & (inh['idx_year'] != 2021), '科研患者编号'])
c1 = child_window((e['phase'] == 'in_index') & e['科研患者编号'].isin(C1), 'in').join(inh.groupby('科研患者编号')['cor_status'].max().rename('w_in'))
c1 = c1[c1.index.isin(C1)]
T7a = pd.crosstab(c1['w_in'].map(WORD), c1['in_best'].map(dict(enumerate(CAT_SHORT))), margins=True).reindex(index=['normal', 'dilation', 'aneurysm', 'All'], columns=CAT_SHORT + ['All'])
T7a.to_csv(os.path.join(OUT_DIR, 'Table7a_C1_inhospital_wording_vs_AHA.csv')); print(T7a)
KEY['C1'] = dict(n=int(len(c1)), wording=c1['w_in'].map(WORD).value_counts().to_dict(), aha=c1['in_best'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict(),
                 aha_aneurysm=int((c1['in_best'] >= 2).sum()), wording_aneurysm=int((c1['w_in'] == 2).sum()),
                 dilation_wording_to_aha=c1.loc[c1['w_in'] == 1, 'in_best'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict(),
                 expected_aneurysm_from_probabilities=float(c1['in_p_aneurysm'].sum()), evidence=c1['in_evidence'].value_counts().to_dict())
print(KEY['C1'])

# first documentation of an aneurysm: AHA (best estimate per report) vs wording
fu = e[e['科研患者编号'].isin(C1) & e['phase'].isin(['in_index', 'post30', 'd31_90', 'd91_365', 'gt365']) & (e['tier'] > 0)].sort_values('检查时间')
fa = fu[fu['aha_best'] >= 2].groupby('科研患者编号').first()
fw = fu[fu['cor_status'] >= 2].groupby('科研患者编号').first()
late_a, late_w = fa[fa['phase'] != 'in_index'], fw[fw['phase'] != 'in_index']
KEY['first_documentation'] = dict(
    aha_ever=int(len(fa)), aha_first_in_hospital=int((fa['phase'] == 'in_index').sum()), aha_first_after_discharge=int(len(late_a)),
    aha_after_by_window=late_a['phase'].value_counts().to_dict(), aha_after_inhospital_category=c1.loc[late_a.index.intersection(c1.index), 'in_best'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict(),
    aha_after_tier_of_first_report=late_a['tier'].value_counts().to_dict(),
    wording_ever=int(len(fw)), wording_first_after_discharge=int(len(late_w)),
    wording_after_inhospital_AHA=c1.loc[late_w.index.intersection(c1.index), 'in_best'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict())
print(KEY['first_documentation'])

# yield of the first post-discharge echo (<=30 d) after a normal in-hospital course
p30 = e[e['科研患者编号'].isin(C1) & (e['phase'] == 'post30') & (e['tier'] > 0)].sort_values('检查时间').groupby('科研患者编号').first()
rows = []
for lab, base in [('wording normal in hospital (v3 definition)', c1.index[c1['w_in'] == 0]), ('AHA no involvement in hospital (Z<2)', c1.index[c1['in_best'] == 0]),
                  ('AHA <2.5 in hospital (none or dilation only)', c1.index[c1['in_best'] <= 1])]:
    q = p30.loc[p30.index.intersection(base)]
    for olab, ok in [('wording lesion (dilation/aneurysm)', q['cor_status'] >= 1), ('AHA >= dilation only (Z>=2)', q['aha_best'] >= 1),
                     ('AHA aneurysm (Z>=2.5)', q['aha_best'] >= 2), ('AHA medium or larger (Z>=5)', q['aha_best'] >= 3)]:
        k_ = int(ok.sum()); lo, hi = wilson(k_, len(q))
        rows.append(dict(baseline=lab, n_with_echo_le30d=len(q), outcome_at_first_postdischarge_echo=olab, events=k_, pct=k_ / len(q) * 100, ci_low=lo, ci_high=hi,
                         events_from_model_tier=int((ok & (q['tier'] == 4)).sum())))
T7b = pd.DataFrame(rows); T7b.round(2).to_csv(os.path.join(OUT_DIR, 'Table7b_first_postdischarge_echo_yield.csv'), index=False); print(T7b.round(2).to_string(index=False))
KEY['first_echo_yield'] = T7b.round(2).to_dict('records')

# in-hospital "dilation" wording: what it was, and what "dilation -> aneurysm progression" was
dil = c1[c1['w_in'] == 1].index
later = e[e['科研患者编号'].isin(dil) & e['phase'].isin(['post30', 'd31_90', 'd91_365', 'gt365']) & (e['tier'] > 0)]
lw = later.groupby('科研患者编号').agg(w_after=('cor_status', 'max'), a_after=('aha_best', 'max'))
prog = lw[lw['w_after'] >= 2].join(c1[['in_best']])
prog['aha_increase'] = prog['a_after'] > prog['in_best']
KEY['dilation_progression'] = dict(followed=int(len(lw)), wording_progression=int(len(prog)),
                                   inhospital_AHA=prog['in_best'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict(),
                                   later_max_AHA=prog['a_after'].map(dict(enumerate(CAT_SHORT))).value_counts().to_dict(),
                                   with_AHA_category_increase=int(prog['aha_increase'].sum()),
                                   from_below_2_5_to_aneurysm=int(((prog['in_best'] <= 1) & (prog['a_after'] >= 2)).sum()))
print(KEY['dilation_progression'])

## 16. Where the evidence is thin: coverage by year and the manual retrieval worklist
Reports from 2025–2026 increasingly put the coronary measurements in a table that the export does not contain ("测值见表"). For those, the
category comes from the wording model unless a note carried the numbers. The worklist lists, for every child whose category in a key
window is uncertain (P(aneurysm) or P(large/giant) between 0.1 and 0.9 without exact numbers), the report that drives the uncertainty, so
that its table values can be looked up in the echo workstation and entered (LMCA/LAD/RCA diameter and Z).

In [ ]:
ab = e[e['cor_status'] >= 1].copy()
ab['evidence'] = np.select([ab['tier'] == 1, ab['tier'] == 2, ab['tier'] == 3, (ab['tier'] == 4) & ab['floor'].notna(), ab['tier'] == 4],
                           ['Tier 1 written Z', 'Tier 2 computed Z', 'Tier 3 note', 'Tier 4 model above numeric floor', 'Tier 4 model only'], 'unclassifiable')
cov = pd.crosstab(ab['yr'], ab['evidence']).reindex(columns=['Tier 1 written Z', 'Tier 2 computed Z', 'Tier 3 note', 'Tier 4 model above numeric floor', 'Tier 4 model only'], fill_value=0)
cov = cov[cov.sum(1) >= 20]
cov.to_csv(os.path.join(OUT_DIR, 'TableS6_evidence_tier_by_year_abnormal_reports.csv')); print(cov)
KEY['coverage_abnormal_reports'] = {int(y): dict(n=int(r.sum()), exact_pct=round(float(r.iloc[:3].sum() / r.sum() * 100), 1)) for y, r in cov.iterrows()}

wl = []
for k_, m_ in [('inhosp', WIN['inhosp']), ('ever', WIN['ever'])]:
    pmed = None
    notnum = PAT[f'{k_}_evidence'] != 'numeric'
    pa, pl = PAT[f'{k_}_p_aneurysm'], PAT[f'{k_}_p_large']
    prio = pd.Series(np.select([notnum & (pa.between(0.1, 0.9) | pl.between(0.1, 0.9))], ['A'], ''), index=PAT.index)
    q = e[m_ & e['科研患者编号'].isin(PAT.index) & (e['tier'] == 4) & (e['cor_status'] >= 1)].copy()
    q['p_medium_plus'] = q['P3'] + q['P4']
    q['exp_cat'] = (q[[f'P{k}' for k in range(5)]].values * np.arange(5)).sum(1)
    q['priority'] = q['科研患者编号'].map(prio)
    q.loc[(q['priority'] == '') & q['p_medium_plus'].between(0.1, 0.9) & q['科研患者编号'].map(notnum), 'priority'] = 'B'
    q = q[q['priority'] != ''].sort_values(['科研患者编号', 'exp_cat', '检查时间'], ascending=[True, False, True]).groupby('科研患者编号').head(1)
    q['window'] = k_
    wl.append(q)
WL = pd.concat(wl).sort_values(['priority', 'window']).drop_duplicates('report_key')
WL = WL[['priority', '科研患者编号', '科研就诊编号', '检查时间', 'window', 'phase', 'cor_status', 'aha_best', 'p_aneurysm', 'p_medium_plus', 'p_large', 'floor', 'report_key', '儿童超声心动图结论']].copy()
WL['cor_status'] = WL['cor_status'].map(WORD); WL['aha_best'] = WL['aha_best'].map(dict(enumerate(CAT_SHORT)))
for c in ['LMCA_mm', 'LMCA_Z', 'LAD_mm', 'LAD_Z', 'LCX_mm', 'RCA_mm', 'RCA_Z', 'height_cm', 'weight_kg', 'entered_by']:
    WL[c] = ''
WL = WL.sort_values(['priority', 'window', 'p_large', 'p_aneurysm'], ascending=[True, True, False, False])
WL.to_csv(os.path.join(OUT_DIR, 'Worklist_reports_for_manual_table_lookup.csv'), index=False, encoding='utf-8-sig')
KEY['worklist'] = dict(reports=int(len(WL)), children=int(WL['科研患者编号'].nunique()), by_priority=WL['priority'].value_counts().to_dict(),
                       by_year=WL['检查时间'].dt.year.value_counts().sort_index().to_dict(), by_window=WL['window'].value_counts().to_dict())
print(KEY['worklist'])

## 17. Figures

In [ ]:
def cochran_armitage(k, n):
    k, n = np.asarray(k, float), np.asarray(n, float); s = np.arange(len(k)); N = n.sum(); p = k.sum() / N
    T_ = np.sum(s * (k - n * p)); V = p * (1 - p) * (np.sum(n * s ** 2) - np.sum(n * s) ** 2 / N)
    return 2 * (1 - stats.norm.cdf(abs(T_ / np.sqrt(V)))) if V > 0 else np.nan


def fmt_p(p):
    return 'P < 0.001' if p < 0.001 else f'P = {p:.3f}'


# Figure 1 - reconstruction of the laboratory's Z equations
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
ax = axs[0]; q = PAIRS[PAIRS['Z'].between(-5, 30)]
ax.scatter(q['Z'], q['Zc'], s=4, color=BLACK, alpha=0.25, lw=0, label=f'{len(q):,} diameter–Z pairs with Z −5 to 30 ({q["rid"].nunique():,} reports)')
ax.plot([-5, 30], [-5, 30], ls='--', color=MGREY, lw=1, label='Line of identity')
ax.set_xlim(-5, 30); ax.set_ylim(-5, 30); ax.set_xlabel('Z-score written in the report'); ax.set_ylabel('Z-score from reconstructed equation')
ax.legend(frameon=False, loc='upper left', fontsize=9); panel(ax, 'A')
ax = axs[1]; g = PAIRS.groupby('bsa_bin', observed=True)['err']
xs = np.arange(g.ngroups); w01 = g.apply(lambda s: (s.abs() < 0.1).mean() * 100).values; w05 = g.apply(lambda s: (s.abs() < 0.5).mean() * 100).values
ax.bar(xs - 0.2, w01, 0.38, color=DGREY, edgecolor=BLACK, label='|difference| < 0.1')
ax.bar(xs + 0.2, w05, 0.38, color='white', edgecolor=BLACK, label='|difference| < 0.5')
labs = [f'{iv.left:g}–{iv.right:g}' if iv.right < 3 else f'>{iv.left:g}' for iv in g.size().index]
ax.set_xticks(xs); ax.set_xticklabels([f'{l}\n(n={n:,})' for l, n in zip(labs, g.size().values)], fontsize=9)
ax.set_xlabel('Body-surface area (m²)'); ax.set_ylabel('Pairs within the margin (%)'); ax.set_ylim(0, 118); ax.set_yticks(range(0, 101, 20))
ax.legend(frameon=False, loc='upper center', ncol=2, fontsize=9); panel(ax, 'B')
fig.tight_layout(); savefig(fig, 'Figure1_reconstructed_Z_equations')

# Figure 2 - what the words mean
fig, axs = plt.subplots(1, 3, figsize=(15, 4.8), gridspec_kw={'width_ratios': [1.1, 1.1, 1]})
ax = axs[0]; order = ['normal', 'upper-normal', 'dilation', 'aneurysm']
bottom = np.zeros(4)
for k in range(5):
    vals = p_all.loc[order, k].values
    ax.bar(range(4), vals, bottom=bottom, color=CAT_COL[k], edgecolor=BLACK, lw=0.6, label=CAT_LAB[k]); bottom += vals
ax.set_xticks(range(4)); ax.set_xticklabels([f'{w.capitalize()}\n(n={int(t_all.loc[w].sum()):,})' for w in order], fontsize=9)
ax.set_xlabel('Sonographer wording'); ax.set_ylabel('Reports (%)'); ax.set_ylim(0, 100)
ax.legend(frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(0.0, -0.2), ncol=2); panel(ax, 'A')
ax = axs[1]; data = [cmp_.loc[cmp_['wording'] == w, 'maxZ'].values for w in order]
bp = ax.boxplot(data, positions=range(4), widths=0.55, whis=(5, 95), showfliers=False, patch_artist=True)
for b in bp['boxes']:
    b.set(facecolor='white', edgecolor=BLACK)
for part in ('whiskers', 'caps', 'medians'):
    for l in bp[part]:
        l.set(color=BLACK)
for thr, l_, lw_ in zip([2, 2.5, 5, 10], [':', '-.', '--', '-'], [1.2, 1, 1, 0.8]):
    ax.axhline(thr, color=MGREY, lw=lw_, ls=l_, label=f'AHA threshold Z = {thr:g}')
ax.set_xticks(range(4)); ax.set_xticklabels([w.capitalize() for w in order], fontsize=9); ax.set_ylim(-4, 42)
ax.plot([], [], ' ', label='Box: IQR and median; whiskers: 5th–95th percentile')
ax.set_xlabel('Sonographer wording'); ax.set_ylabel('Maximal coronary Z-score')
ax.legend(frameon=False, fontsize=8, loc='upper left'); panel(ax, 'B')
ax = axs[2]; dy = dil_year.copy()
ci = np.array([wilson(int(round(p_ / 100 * n_)), int(n_)) for p_, n_ in zip(dy['pct_AHA_aneurysm'], dy['n'])])
ax.errorbar(dy.index, dy['pct_AHA_aneurysm'], yerr=[dy['pct_AHA_aneurysm'] - ci[:, 0], ci[:, 1] - dy['pct_AHA_aneurysm']], color=BLACK, marker='o', capsize=3,
            label='"Dilation" reports that are AHA aneurysms (95% CI)')
ax.set_ylim(0, 100); ax.set_xlabel('Year of echocardiogram'); ax.set_ylabel('AHA aneurysm, Z ≥ 2.5 (%)')
ax.set_xticks(dy.index); ax.set_xticklabels([str(int(y)) for y in dy.index], rotation=45, fontsize=9)
ax.legend(frameon=False, fontsize=8, loc='lower right'); panel(ax, 'C')
fig.tight_layout(); savefig(fig, 'Figure2_wording_versus_AHA')

# Figure 3 - external anchors
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, tab, xl, let in [(axs[0], by_aha, 'AHA category (whole follow-up)', 'A'), (axs[1], by_word, 'Report wording (whole follow-up)', 'B')]:
    xs = np.arange(len(tab))
    kw = (tab['warfarin_pct'] / 100 * tab['n']).round().astype(int).values; kc = (tab['clopidogrel_pct'] / 100 * tab['n']).round().astype(int).values
    ax.bar(xs - 0.2, tab['warfarin_pct'], 0.38, color=BLACK, label=f'Warfarin (trend {fmt_p(cochran_armitage(kw, tab["n"].values))})')
    ax.bar(xs + 0.2, tab['clopidogrel_pct'], 0.38, color=LGREY, edgecolor=BLACK, label=f'Clopidogrel (trend {fmt_p(cochran_armitage(kc, tab["n"].values))})')
    ax.set_xticks(xs); ax.set_xticklabels([f'{str(i).capitalize()}\n(n={int(n):,})' for i, n in zip(tab.index, tab['n'])], fontsize=8.5)
    ax.set_xlabel(xl); ax.set_ylabel('Children treated (%)'); ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=8.5, loc='upper left'); panel(ax, let)
ax = axs[2]; T4n = T4.fillna(0); cols_ = ['coded aneurysm', 'coded dilation', 'no coronary code']; colr = [BLACK, MGREY, 'white']
bottom = np.zeros(len(T4n))
for c, col in zip(cols_, colr):
    vals = (T4n.get(c, 0) / T4n.sum(1) * 100).values
    ax.bar(range(len(T4n)), vals, bottom=bottom, color=col, edgecolor=BLACK, lw=0.6, label=c.capitalize()); bottom += vals
ax.set_xticks(range(len(T4n))); ax.set_xticklabels([f'{i}\n(n={int(n):,})' for i, n in zip(T4n.index, T4n.sum(1))], fontsize=8.5)
ax.set_xlabel('AHA category in hospital'); ax.set_ylabel('Index-admission discharge coding (%)'); ax.set_ylim(0, 100)
ax.legend(frameon=False, fontsize=8.5, loc='upper left', bbox_to_anchor=(0, -0.22), ncol=3); panel(ax, 'C')
fig.tight_layout(); savefig(fig, 'Figure3_treatment_and_coding_anchors')

# Figure 4 - impact on cohort C1
fig, axs = plt.subplots(1, 3, figsize=(18, 5.0), gridspec_kw={'width_ratios': [1.05, 0.85, 1.25]})
ax = axs[0]; tb = pd.crosstab(c1['w_in'].map(WORD), c1['in_best']).reindex(index=['normal', 'dilation', 'aneurysm'], columns=range(5), fill_value=0)
pc = tb.div(tb.sum(1), axis=0) * 100; bottom = np.zeros(3)
for k in range(5):
    ax.bar(range(3), pc[k].values, bottom=bottom, color=CAT_COL[k], edgecolor=BLACK, lw=0.6, label=CAT_LAB[k]); bottom += pc[k].values
ax.set_xticks(range(3)); ax.set_xticklabels([f'{w.capitalize()}\n(n={int(tb.loc[w].sum()):,})' for w in tb.index], fontsize=9)
ax.set_xlabel('Maximal in-hospital wording'); ax.set_ylabel('Children (%)'); ax.set_ylim(0, 100)
ax.legend(frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(0, -0.2), ncol=2); panel(ax, 'A')
# B: aneurysms first documented after discharge, by definition, split by the in-hospital AHA category
ax = axs[1]
sets = {'Report wording': late_w.index.intersection(c1.index), 'AHA 2017': late_a.index.intersection(c1.index)}
for j, (lab, ids) in enumerate(sets.items()):
    cnt = c1.loc[ids, 'in_best'].value_counts().reindex(range(5), fill_value=0); bottom = 0
    for k in range(5):
        ax.bar(j, cnt[k], bottom=bottom, color=CAT_COL[k], edgecolor=BLACK, lw=0.6, width=0.6); bottom += cnt[k]
    ax.text(j, bottom + 1, f'n = {int(bottom)}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(2)); ax.set_xticklabels(['Defined by\nreport wording', 'Defined by\nAHA 2017'], fontsize=9.5)
ax.set_xlim(-0.6, 1.6); ax.set_ylim(0, max(len(v_) for v_ in sets.values()) * 1.15); ax.set_ylabel('Aneurysms first found after discharge (n)')
ax.set_xlabel('Color = AHA category in hospital (legend in A)', fontsize=9, fontweight='normal'); panel(ax, 'B')
# C: yield of the first post-discharge echo (<=30 d) after a normal in-hospital course, by outcome definition
ax = axs[2]; yb = T7b[T7b['baseline'].isin(['wording normal in hospital (v3 definition)', 'AHA no involvement in hospital (Z<2)'])]
outs = ['wording lesion (dilation/aneurysm)', 'AHA >= dilation only (Z>=2)', 'AHA aneurysm (Z>=2.5)', 'AHA medium or larger (Z>=5)']
olab = ['Worded\ndilation/aneurysm', 'AHA Z ≥ 2', 'AHA Z ≥ 2.5', 'AHA Z ≥ 5']
for j, (bl, col, lab) in enumerate([('wording normal in hospital (v3 definition)', 'white', 'Normal in hospital by wording'),
                                     ('AHA no involvement in hospital (Z<2)', DGREY, 'Normal in hospital by AHA (Z < 2)')]):
    q = yb[yb['baseline'] == bl].set_index('outcome_at_first_postdischarge_echo').loc[outs]
    xs = np.arange(4) + (j - 0.5) * 0.38
    ax.bar(xs, q['pct'], 0.36, color=col, edgecolor=BLACK, label=f"{lab} (n = {int(q['n_with_echo_le30d'].iloc[0]):,})")
    ax.errorbar(xs, q['pct'], yerr=[q['pct'] - q['ci_low'], q['ci_high'] - q['pct']], fmt='none', ecolor=BLACK, capsize=3, lw=1)
ax.set_xticks(range(4)); ax.set_xticklabels(olab, fontsize=9); ax.set_ylabel('New finding at first echo ≤ 30 d (%)')
ax.set_xlabel('Finding (error bars: 95% CI)', fontsize=10); ax.legend(frameon=False, fontsize=8.5, loc='upper right'); panel(ax, 'C')
fig.tight_layout(w_pad=3.0); savefig(fig, 'Figure4_impact_on_cohort_C1')

# Figure S1 - evidence tiers by year (abnormal-worded reports)
fig, ax = plt.subplots(figsize=(8.5, 4.4)); bottom = np.zeros(len(cov)); colr = [BLACK, DGREY, MGREY, LGREY, 'white']
for c, col in zip(cov.columns, colr):
    ax.bar(range(len(cov)), cov[c].values, bottom=bottom, color=col, edgecolor=BLACK, lw=0.6, label=c); bottom += cov[c].values
ax.set_xticks(range(len(cov))); ax.set_xticklabels([str(int(y)) for y in cov.index]); ax.set_xlabel('Year of echocardiogram')
ax.set_ylabel('Reports worded dilation or aneurysm'); ax.legend(frameon=False, fontsize=8.5, loc='upper left')
fig.tight_layout(); savefig(fig, 'FigureS1_evidence_tier_by_year')
print('figures written')

## 18. Results lock and exports

In [ ]:
fd, eqk, nk = KEY['first_documentation'], KEY['equations'], KEY['numeric']
wv = KEY['wording_vs_aha']
mv = pd.DataFrame(MV).set_index('set')
L_ = []
L_.append(f"RESULTS LOCK — coronary reclassification v1 ({pd.Timestamp.today().date()}) — computed by KD_coronary_AHA_v1.ipynb from 川崎病重新导出30天_1/2/3.xlsx (data end {DATA_END.date()}).")
L_.append('AHA 2017 categories: none Z<2; dilation only 2–<2.5; small 2.5–<5; medium 5–<10; large/giant Z≥10 or ≥8 mm (maximal LMCA/LAD/RCA Z; written LCX Z counted).\n')
L_.append('C0. Data')
L_.append(f"- Echo reports {nk['reports']:,}; with a numeric category {nk['with_numeric']:,} (complete {nk['complete']:,}; floor only {nk['lower_bound']:,}); maximal Z written by the sonographer in {nk['max_from_written_Z']:,}, computed from a written diameter in {nk['max_from_computed_Z']:,}.")
L_.append(f"- Decimal-point corrections {nk['decimal_fixes']}; implausible integer diameters dropped {nk['integer_values_dropped']}.")
L_.append('C1. The laboratory\'s Z equations can be reconstructed')
L_.append(f"- {eqk['n_pairs']:,} diameter–Z pairs from {eqk['n_reports']:,} reports printing height and weight; Stevenson BSA; mean = a + b·√BSA + c·BSA, constant SD.")
L_.append(f"- Computed vs written Z: median |difference| {eqk['median_abs_error']:.3f}; within 0.1 in {eqk['pct_within_0_1']:.1f}%, within 0.5 in {eqk['pct_within_0_5']:.1f}% (BSA > 1.4 m², outside the laboratory's stated reference range: {eqk['pct_within_0_5_bsa_gt_1_4']:.1f}% within 0.5).")
for r in FITS.itertuples():
    L_.append(f"  {r.vessel}: a {r.a:.4f}, b {r.b_sqrtBSA:.4f}, c {r.c_BSA:.4f}, SD {r.SD_mm:.4f} mm (n = {r.n_pairs:,} pairs, {r.years}).")
bf = KEY['bsa_fallback']
L_.append(f"- Age–sex median BSA used as last resort: median |ΔZ| {bf['median_abs_error']:.2f} (P90 {bf['p90_abs_error']:.2f}); aneurysm (Z ≥ 2.5) agreement {bf['aneurysm_agreement']:.1f}%.")
L_.append('C2. What the words mean (reports with a complete numeric category; % none / dilation only / small / medium / large-giant)')
for w in ['normal', 'upper-normal', 'dilation', 'aneurysm']:
    d_ = wv[w]
    L_.append(f"- '{w}' (n = {d_['n']:,}): {' / '.join(f'{x:.1f}' for x in d_['pct'])}; median maximal Z {d_['median_Z']} (IQR {d_['IQR'][0]}–{d_['IQR'][1]}). Written-Z subset (n = {d_['n_writtenZ']:,}): {' / '.join(f'{x:.1f}' for x in d_['pct_writtenZ'])}.")
L_.append(f"- 'Dilation' reports that are AHA aneurysms, by year: {', '.join(f'{int(k)} {v:.1f}%' for k, v in KEY['dilation_wording_aneurysm_by_year'].items())}.")
L_.append('C3. Evidence tiers (reports): ' + ', '.join(f"tier {k}: {v:,}" for k, v in KEY['tiers'].items()) + f" (0 = no coronary statement and no number); note values contradicting their report's wording, not used: {KEY['tier3_conflicts_not_used']}.")
L_.append('- Exact evidence (tiers 1–3) among reports worded dilation/aneurysm, by year: ' + ', '.join(f"{y} {d['exact_pct']:.1f}% of {d['n']:,}" for y, d in KEY['coverage_abnormal_reports'].items()) + '.')
for r in KEY['tier3']['validation']:
    L_.append(f"- Tier 3 linkage ({r['how']}): exact AHA agreement {r['exact_AHA_agreement']:.1f}%, aneurysm agreement {r['aneurysm_agreement']:.1f}% (n = {int(r['n'])} reports that also have their own numbers).")
L_.append('C4. Wording model (Tier 4)')
for s_ in mv.index:
    r = mv.loc[s_]
    L_.append(f"- {s_}: n {int(r['n']):,}; exact 5-category accuracy {r['exact_accuracy']:.1f}%; weighted kappa {r['weighted_kappa']:.3f}; AUC aneurysm {r['auc_aneurysm']:.3f}; AUC large {r['auc_large']:.3f}; aneurysm yes/no accuracy {r['accuracy_aneurysm_yes_no']:.1f}%; observed/predicted aneurysm {r['observed_aneurysm_pct']:.1f}%/{r['predicted_aneurysm_pct']:.1f}%.")
L_.append('- Specification sensitivity, 2025–2026 reports without numbers (mean predicted probability, primary vs alternative model without a 2025–26 era term): ' +
          '; '.join(f"{r['wording']} (n = {r['n']}) {r['model']}: P(aneurysm) {r['mean_P_aneurysm']:.2f}, P(medium+) {r['mean_P_medium_plus']:.2f}, P(large) {r['mean_P_large']:.2f}" for r in KEY['tier4_sensitivity_2025_2026']) + '.')
if 'audits' in KEY:
    L_.append('C5. Blinded manual extraction audits')
    for r in KEY['audits']:
        L_.append(f"- {r['sample']} (n = {r['n']}): all seven numeric fields agree {r['all_seven_fields']} ({r['all_seven_pct']:.1f}%); per-vessel diameter {r['LMCA_diameter_pct']:.1f}/{r['LAD_diameter_pct']:.1f}/{r['LCX_diameter_pct']:.1f}/{r['RCA_diameter_pct']:.1f}%; written Z {r['LMCA_Z_pct']:.1f}/{r['LAD_Z_pct']:.1f}/{r['RCA_Z_pct']:.1f}%; unmeasured-segment flag {r['unmeasured_flag_agree']} (missed {r['unmeasured_missed_by_extractor']}, over-called {r['unmeasured_overcalled_by_extractor']}).")
    if 'audit_log_first_look' in KEY:
        for k_, d_ in KEY['audit_log_first_look'].items():
            if isinstance(d_, dict):
                L_.append(f"  first look ({k_}, before the fixes it prompted): all seven fields {d_.get('first_look_all_seven_fields')}; flag {d_.get('first_look_unmeasured_flag', d_.get('first_look_unmeasured_flag_among_reports_with_numbers'))}.")
L_.append('C6. External anchors (children, whole follow-up)')
for i_, r in T3.round(1).iterrows():
    L_.append(f"- {i_[0]} = {i_[1]}: n {int(r['n']):,}; warfarin {r['warfarin_pct']:.1f}%; clopidogrel {r['clopidogrel_pct']:.1f}%; LMWH {r['lmwh_pct']:.1f}%.")
L_.append(f"- Warfarin-treated children (n = {KEY['warfarin']['n']}): AHA {KEY['warfarin']['aha']}; wording {KEY['warfarin']['wording']}.")
L_.append('- Index discharge coding by in-hospital AHA category: ' + '; '.join(f"{c} n {d['n']:,}: coded aneurysm {d['coded_aneurysm_pct']:.1f}%, coded dilation {d['coded_dilation_pct']:.1f}%" for c, d in KEY['coding'].items()) + '.')
if 'master' in KEY:
    mk = KEY['master']
    L_.append('C7. Frozen master v1.3 (reference only)')
    L_.append(f"- Overlap {mk['overlap']:,} children; exact agreement (0/1/2) {mk['exact_agreement_pct']:.1f}%.")
    L_.append(f"- Master 'qualitative dilation' (n = {mk['qualitative_dilation']['n']}): this pipeline {mk['qualitative_dilation']['ours']} (0 = <2, 1 = 2–<2.5, 2 = ≥2.5); {mk['qualitative_dilation']['all_inhospital_reports_normal']} have only normal-worded in-hospital reports in this export ({mk['qualitative_dilation']['of_which_preadmission_echo_in_notes']} with coronary dilation from a pre-admission echo quoted in the admission narrative, {mk['qualitative_dilation']['of_which_coded_dilation']} coded 冠状动脉扩张).")
    L_.append(f"- Master class 1 (n = {mk['master_class1']['n']}): {mk['master_class1']['ours']}; master class 2 (n = {mk['master_class2']['n']}): {mk['master_class2']['ours']}.")
ck = KEY['C1']
L_.append('C8. Impact on the locked surveillance results (cohort C1)')
L_.append(f"- C1 n = {ck['n']:,}. In-hospital maximum by wording {ck['wording']}; by AHA {ck['aha']} (evidence {ck['evidence']}).")
L_.append(f"- In-hospital aneurysm: wording {ck['wording_aneurysm']} vs AHA {ck['aha_aneurysm']} (expected from probabilities {ck['expected_aneurysm_from_probabilities']:.0f}); in-hospital 'dilation' wording → AHA {ck['dilation_wording_to_aha']}.")
L_.append(f"- Children ever with an AHA aneurysm {fd['aha_ever']}: first documented in hospital {fd['aha_first_in_hospital']}, after discharge {fd['aha_first_after_discharge']} {fd['aha_after_by_window']}; in-hospital category of the latter {fd['aha_after_inhospital_category']}; tier of the first aneurysm report {fd['aha_after_tier_of_first_report']}.")
L_.append(f"- Wording-defined post-discharge-first aneurysms {fd['wording_first_after_discharge']} (of {fd['wording_ever']}): in-hospital AHA category {fd['wording_after_inhospital_AHA']}.")
dp = KEY['dilation_progression']
L_.append(f"- In-hospital 'dilation' followed after discharge {dp['followed']}; later worded aneurysm {dp['wording_progression']}: in-hospital AHA {dp['inhospital_AHA']}, later maximal AHA {dp['later_max_AHA']}; AHA category increase {dp['with_AHA_category_increase']}; from <2.5 to aneurysm {dp['from_below_2_5_to_aneurysm']}.")
for r in KEY['first_echo_yield']:
    L_.append(f"- First post-discharge echo ≤30 d | {r['baseline']} (n = {r['n_with_echo_le30d']:,}) | {r['outcome_at_first_postdischarge_echo']}: {r['events']} ({r['pct']:.2f}%; 95% CI {r['ci_low']:.2f}–{r['ci_high']:.2f}; from model tier {r['events_from_model_tier']}).")
L_.append('C9. Manual table look-up worklist')
L_.append(f"- {KEY['worklist']['reports']} reports ({KEY['worklist']['children']} children) {KEY['worklist']['by_priority']} (A = aneurysm yes/no or large/giant uncertain; B = size small vs medium+ uncertain); by year {KEY['worklist']['by_year']}.")
open(os.path.join(OUT_DIR, 'Results_locked_coronary_v1.txt'), 'w', encoding='utf-8').write('\n'.join(L_))
print('\n'.join(L_))

# per-report and per-child tables (research IDs only; no names)
rep_cols = ['report_key', '科研患者编号', '科研就诊编号', '检查时间', 'phase', 'cor_status', 'upper_normal'] + \
           [f'{v}_{k}' for v in VESSELS for k in ('Dclean', 'Zfinal', 'Zsrc')] + \
           ['maxZ', 'maxZ_vessel', 'maxZ_src', 'maxD', 'BSA', 'BSA_src', 'num_status', 'h_aha', 'h_how', 'tier', 'floor', 'P0', 'P1', 'P2', 'P3', 'P4', 'aha_best', 'p_aneurysm', 'p_large']
REP = e[rep_cols].copy(); REP['cor_status'] = REP['cor_status'].map(WORD)
PAT['C1_surveillance_cohort'] = PAT.index.isin(C1)
REP.to_csv(os.path.join(OUT_DIR, 'per_report_AHA_classification.csv'), index=False, encoding='utf-8-sig')
PAT.to_csv(os.path.join(OUT_DIR, 'per_child_AHA_classification.csv'), encoding='utf-8-sig')
json.dump(KEY, open(os.path.join(OUT_DIR, 'key_numbers.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
names = [c for c in XT.columns if not c.startswith('w_')] + [f'w_{t}_{k}' for t in ('c', 'f') for k in (-1, 0, 1, 2)] + [f'era_{a}_{b}' for a, b in ((2018, 2019), (2020, 2021), (2022, 2024), (2025, 2026))]
pd.DataFrame(WM, index=['intercept'] + names, columns=CAT_SHORT).round(4).to_csv(os.path.join(OUT_DIR, 'TableS7_wording_model_coefficients.csv'))
IDX = {
    'Figure1_reconstructed_Z_equations': 'A, Z computed with the reconstructed laboratory equations vs the Z written in the same report; B, share of pairs within 0.1 and 0.5 Z by body-surface area.',
    'Figure2_wording_versus_AHA': 'A, AHA category by sonographer wording (reports with complete numbers); B, maximal Z by wording with AHA thresholds; C, "dilation" reports that are AHA aneurysms, by year.',
    'Figure3_treatment_and_coding_anchors': 'A/B, warfarin and clopidogrel use by AHA category vs by wording (children, whole follow-up; Cochran-Armitage trend); C, index discharge coding by in-hospital AHA category.',
    'Figure4_impact_on_cohort_C1': 'A, in-hospital wording vs AHA category (cohort C1); B, aneurysms first documented after discharge by definition, colored by in-hospital AHA category; C, yield of the first post-discharge echo by definition.',
    'FigureS1_evidence_tier_by_year': 'Evidence tier of reports worded dilation/aneurysm, by year.',
    'Table1_reconstructed_Z_equations.csv': 'Fitted coefficients and agreement with written Z, per vessel.',
    'Table2_wording_vs_AHA_reports.csv': 'Report wording vs AHA category (all complete reports and written-Z subset).',
    'Table3_treatment_anchor.csv': 'Warfarin/clopidogrel/LMWH by AHA category and by wording.',
    'Table4_discharge_coding_vs_AHA.csv': 'Index discharge coding by in-hospital AHA category.',
    'Table5_master_v13_crosswalk.csv': 'Frozen master v1.3 Coronary_Severity_3 by severity_source vs this pipeline (reference only).',
    'Table6_blinded_audits.csv': 'Agreement of the extractor with blinded manual reading (three samples).',
    'Table7a_C1_inhospital_wording_vs_AHA.csv': 'Cohort C1: maximal in-hospital wording vs AHA category.',
    'Table7b_first_postdischarge_echo_yield.csv': 'First post-discharge echo (<=30 d) after a normal in-hospital course: wording vs AHA definitions.',
    'TableS1_reference_BSA_by_age_sex.csv': "Cohort's own age-sex median BSA.",
    'TableS2a_equation_error_by_BSA.csv / TableS2b_equation_error_by_year.csv': 'Equation error by BSA and by year.',
    'TableS3_note_linkage_accuracy.csv': 'Tier 3 linkage accuracy on reports with their own numbers.',
    'TableS4_dilation_wording_by_year.csv': '"Dilation" reports that are AHA aneurysms, by year.',
    'TableS5_audit_*_merged.csv': 'Report-level audit comparison (reader vs extractor).',
    'TableS6_evidence_tier_by_year_abnormal_reports.csv': 'Evidence tiers by year.',
    'TableS7_wording_model_coefficients.csv': 'Coefficients of the wording model (multinomial logistic).',
    'TableS8_tier4_specification_sensitivity_2025_2026.csv': 'Mean predicted probabilities for 2025-2026 reports without numbers under the primary and an alternative model specification.',
    'per_report_AHA_classification.csv': 'Every report: per-vessel diameter/Z/source, BSA and source, numeric status, tier, category distribution P0-P4, best category.',
    'per_child_AHA_classification.csv': 'Every child: best category, P(aneurysm), P(large/giant), evidence, n reports for in hospital / <=30 d / whole follow-up; wording maxima; treatment anchors; coding; C1 flag.',
    'Worklist_reports_for_manual_table_lookup.csv': 'Reports whose unexported measurement table would settle an uncertain child-level category (priority A/B), with blank columns to enter values.',
    'Results_locked_coronary_v1.txt / key_numbers.json': 'All key numbers (computed).'}
open(os.path.join(OUT_DIR, 'Figures_and_tables_index.txt'), 'w', encoding='utf-8').write('\n'.join(f'{k}: {v}' for k, v in IDX.items()))
print('exported:', sorted(os.listdir(OUT_DIR)))